# NeurIPS Diffusion Evaluation

Train diffusion model, compute FID (AE or GyroSwin latent space), evaluate warm restarts, test FID-vs-convergence.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, os

sys.path.append("..")
os.environ["CUDA_VISIBLE_DEVICES"] = "7"

In [ ]:
import omegaconf, yaml
from collections import defaultdict

import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from scipy.stats import pearsonr
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

from neugk.diffusion import get_diffusion_runner

from neurips_diff_eval import (
    compute_statistics,
    compute_fid,
    extract_gyroswin_latents,
    to_model_space,
    from_model_space,
)

## 0. Configuration

In [ ]:
DATA_PATH = "/local00/bioinf/galletti/preprocessed_kvikio"
AE_CHECKPOINT = "/restricteddata/ukaea/checkpoints/neurips26/AE_noCond/20260405_022851_327/best.pth"
# Same scaling-law GyroSwin checkpoint used by neurips_fid_gyroswin_latents.ipynb
GYROSWIN_CHECKPOINT = "/restricteddata/ukaea/checkpoints/scaling_law/gyroswin_xxl_fluxavg_cond_nodrop_l1"
GKW_RAW_DIR = "/restricteddata/ukaea/gyrokinetics/raw"

# pretrained diffusion model
PRETRAINED_DIR = "/restricteddata/ukaea/checkpoints/neurips26/DIFF_FLOW/20260412_180101_948/"

ID_VAL = [
    "iteration_262.h5",
    "iteration_135.h5",
    "iteration_8.h5",
    "iteration_232.h5",
    "iteration_148.h5",
    "iteration_115.h5",
]
OOD_VAL = [f"ood_iteration_{i}.h5" for i in range(5)]
TRAIN_TRAJS = "iteration_{0-5,7-12,14-31,33-82,84-99}.h5"

N_EPOCHS = 10
BATCH_SIZE = 256
LR = 2e-3
VAL_EVERY = 30
N_DENOISING_STEPS = 15

MINIBATCH_OT = True
NOISE_DISTRIBUTION = "gaussian"  # "gaussian" or "mixture"
CONTINUOUS_TIME = True

FID_N_COMPONENTS = 512
FID_FRAC = 0.5             # fraction of the val set used for FID feature collection
VAL_SUBSAMPLE = 10         # runner valset stride (1=full, 5≈20%); used by runner.evaluate / probes
GEN_BATCH_SIZE = 32

In [ ]:
# --- paper-figure helpers (style + save/load) -----------------------------
from neurips_paper_plots import (
    apply_paper_style, save_fig, save_method_results,
    pretty_scatter, COLOR_FLUX, COLOR_KY, COLOR_QY, COLOR_REF, COLOR_GYROFLOW, darken,
)

apply_paper_style()
RESULTS_DIR = "/system/user/galletti/git/neural-gyrokinetics-gitlab/notebooks/figs/paper"
METHOD_NAME = "GyroFlow"
print(f"paper figures will be saved to {RESULTS_DIR}")


## 1. Train Diffusion Model

In [ ]:
print(f"loading pretrained from {PRETRAINED_DIR}")
pretrained_cfg = omegaconf.OmegaConf.load(os.path.join(PRETRAINED_DIR, "config.yaml"))
pretrained_cfg.output_path = PRETRAINED_DIR
pretrained_cfg.dataset.path = DATA_PATH
pretrained_cfg.ae_checkpoint = os.path.dirname(AE_CHECKPOINT)
pretrained_cfg.dataset.gds_override = True
pretrained_cfg.dataset.validation_trajectories = ID_VAL
pretrained_cfg.dataset.val_subsample = VAL_SUBSAMPLE
pretrained_cfg.dataset.eval_cond_filters = {}
pretrained_cfg.validation.probe = {"targets": []}
pretrained_cfg.logging.writer = None
pretrained_cfg.logging.tqdm = True
pretrained_cfg.training.num_workers = 0
pretrained_cfg.training.pin_memory = False
pretrained_cfg.ddp.enable = False
pretrained_cfg.deepspeed.enable = False
runner = get_diffusion_runner(rank=0, cfg=pretrained_cfg, world_size=1)
torch.use_deterministic_algorithms(False)
ckp = torch.load(
    os.path.join(PRETRAINED_DIR, "best.pth"),
    map_location=runner.device,
    weights_only=False,
)
runner.model.load_state_dict(ckp["model_state_dict"])
runner.model.eval()
print(f"--> loaded diffusion weights from epoch {ckp.get('epoch', '?')}")
print(f"Model: {sum(p.numel() for p in runner.model.parameters())/1e6:.1f}M params")
print(f"Train: {len(runner.trainset)}, Val: {sum(len(v) for v in runner.valsets)}")


In [ ]:
raw = runner.trainset.__getitem__(100, get_normalized=False, override_latens=True)
print(f"raw std: {raw.df.std():.4f}, raw mean: {raw.df.mean():.4f}")
scale, shift = runner.trainset._get_scale_shift(0, "df", raw.df)
print(f"shift shape: {shift.shape}, scale shape: {scale.shape}")
print(f"shift: {shift.squeeze()}")
print(f"scale: {scale.squeeze()}")
manual = (raw.df - shift) / scale
print(f"manual normalized std: {manual.std():.4f}")

In [ ]:
from neugk.plot_utils import plot_nd

sample = runner.trainset.__getitem__(100, get_normalized=True, override_latens=True)
df_in = sample.df.unsqueeze(0).to(runner.device)
cond = sample.conditioning.unsqueeze(0).to(runner.device)

ae = runner.autoencoder
ae.eval()
with torch.no_grad():
    recon = ae(df_in, condition=cond)["df"]

df_gt = df_in[0].cpu()
df_rec = recon[0].cpu()

print(f"Input shape: {df_gt.shape}, Recon shape: {df_rec.shape}")
print(f"AE recon MSE: {(df_gt - df_rec).pow(2).mean():.6f}")
print(f"AE recon rel err: {(df_gt - df_rec).norm() / df_gt.norm():.4f}")

_ = plot_nd(df_gt, df_rec, to_wandb=False)

In [ ]:
sample = runner.trainset.__getitem__(0, get_normalized=True, override_latens=True)
cond = sample.conditioning.unsqueeze(0).to(runner.device)

runner.model.eval()
with torch.no_grad():
    gen_out = runner.sample(cond, latent_only=False, steps=N_DENOISING_STEPS)

df_gt = sample.df.cpu()
df_gen = gen_out["df"][0].cpu()

print(f"GT shape: {df_gt.shape}, Gen shape: {df_gen.shape}")

_ = plot_nd(df_gt, df_gen, to_wandb=False)

## 1.5 Flux UQ (ID)

In [ ]:
runner.model.eval()
log_metrics, val_plots, _ = runner.evaluate(epoch=0, evaluate_probing=False, no_save=True)

In [ ]:
val_plots.keys()

In [ ]:
for k, v in sorted(log_metrics.items()):
    print(f"  {k}: {v:.4f}")

display(val_plots["avg_flux_UQ"].image)

In [ ]:
from neurips_diff_eval import (
    collect_latents,
    generate_latents,
    fit_probes,
    plot_probes,
    encode_valset,
    plot_val_probe,
)

PROBE_N_COMPONENTS = 256
PROBE_ALPHA = 1.0
PROBE_SUBSAMPLE_FRAC = 0.10   # 10% of the training latents

_cond_keys = sorted(runner.cfg.model.conditioning)
X_ae, y_flux, C_train = collect_latents(runner.trainset.precomputed_latents, _cond_keys)
print(f"training set: {X_ae.shape[0]} samples, latent dim={X_ae.shape[1]}")

PROBE_SUBSAMPLE = max(1, int(PROBE_SUBSAMPLE_FRAC * len(X_ae)))
if PROBE_SUBSAMPLE and PROBE_SUBSAMPLE < len(X_ae):
    idx = np.random.choice(len(X_ae), PROBE_SUBSAMPLE, replace=False)
    X_ae, y_flux, C_train = X_ae[idx], y_flux[idx], C_train[idx]
    print(f"subsampled to {PROBE_SUBSAMPLE} samples")

X_gen = generate_latents(runner, C_train, batch_size=GEN_BATCH_SIZE, steps=N_DENOISING_STEPS)

probes = fit_probes(
    X_ae,
    X_gen,
    y_flux,
    _cond_keys,
    C_train,
    n_components=PROBE_N_COMPONENTS,
    alpha=PROBE_ALPHA,
)
print(
    f"PCA: {X_ae.shape[1]} -> {probes['pca'].n_components_} ({probes['pca'].explained_variance_ratio_.sum():.1%} var)"
)
print(f"flux probe — RMSE ae: {probes['flux']['rmse_ae']:.4f}, RMSE gen: {probes['flux']['rmse_gen']:.4f}")
for k in _cond_keys:
    print(f"  {k} — RMSE ae: {probes['cond']['rmse'][k]['ae']:.4f}, RMSE gen: {probes['cond']['rmse'][k]['gen']:.4f}")

_ = plot_probes(y_flux, probes, _cond_keys, C_train, X_ae, X_gen)

X_val_ae, C_val, y_val_gt, val_fi = encode_valset(
    runner.valsets[0],
    runner.autoencoder,
    _cond_keys,
    runner.device,
    batch_size=GEN_BATCH_SIZE,
)
X_val_gen = generate_latents(runner, C_val, batch_size=GEN_BATCH_SIZE, steps=N_DENOISING_STEPS)

pca = probes["pca"]
pred_val_ae = probes["flux"]["probe_ae"].predict(pca.transform(X_val_ae))
pred_val_gen = probes["flux"]["probe_gen"].predict(pca.transform(X_val_gen))
rmse_val_ae = np.sqrt(((y_val_gt - pred_val_ae) ** 2).mean())
rmse_val_gen = np.sqrt(((y_val_gt - pred_val_gen) ** 2).mean())
print(f"flux probe (val) — RMSE ae: {rmse_val_ae:.4f}, RMSE gen: {rmse_val_gen:.4f}")

_ = plot_val_probe(
    runner.valsets[0],
    pred_val_ae,
    pred_val_gen,
    y_val_gt,
    val_fi,
    rmse_val_ae,
    rmse_val_gen,
)

In [ ]:
# === paper figure: flux probe (generative latents only) ====================
# Uses the val-set predictions already computed above:
#   y_val_gt        — ground-truth scalar flux per val sample
#   pred_val_gen    — diffusion-latent → flux probe prediction
# Outputs:  <RESULTS_DIR>/probe_flux_gen.{pdf,png}
import numpy as np

apply_paper_style()
fig, ax = plt.subplots(figsize=(3.4, 3.4))
yt = np.asarray(y_val_gt, dtype=float)
yp = np.asarray(pred_val_gen, dtype=float)
mask = np.isfinite(yt) & np.isfinite(yp)
yt, yp = yt[mask], yp[mask]
rmse = float(np.sqrt(np.mean((yt - yp) ** 2)))
r = float(np.corrcoef(yt, yp)[0, 1]) if len(yt) > 2 else float("nan")
pretty_scatter(
    ax, yt, yp, color=COLOR_GYROFLOW, label="diffusion latents",
    diag=True, annot=f"RMSE = {rmse:.3f}\nr = {r:.3f}",
)
ax.set_xlabel("ground-truth Q")
ax.set_ylabel("predicted Q (probe on diff latents)")
ax.set_title("Flux probe — generative latents")
fig.tight_layout()
paths = save_fig(fig, "probe_flux_gen", RESULTS_DIR)
print(f"saved: {paths}")
plt.show()

PAPER_PROBE = {
    "y_true": yt, "y_pred_gen": yp, "rmse": rmse, "pearson": r,
}


## 2. FID Evaluation

- **`ae`**: FID in AE bottleneck (fast, no decode)
- **`gyroswin`**: FID using frozen GyroSwin encoder (Inception-FID analog)

In [ ]:
import pickle as _pickle
from neurips_gyroswin_eval import load_gyroswin_model
from neugk.dataset.cyclone_diff import CycloneAEDataset
from neugk.dataset.backend import KvikIOBackend

gyroswin_model, gs_cfg, _ = load_gyroswin_model(
    GYROSWIN_CHECKPOINT, dataset=runner.trainset, device=runner.device,
)
feature_fn = lambda b, device, **kw: extract_gyroswin_latents(gyroswin_model, b, device, **kw)
fid_label = "GyroSwin-FID"

_gs_stats_path = os.path.join(GYROSWIN_CHECKPOINT, "normalization_stats.pkl")
with open(_gs_stats_path, "rb") as f:
    gs_stats = _pickle.load(f)
print(f"loaded GyroSwin normalization stats from {_gs_stats_path}")

# Per-field zscore dict (CycloneAEDataset expects this shape, not a bare
gs_norm = {
    "df":   {"type": "zscore", "agg_axes": [1, 2, 3, 4, 5]},
    "flux": {"type": "zscore", "agg_axes": None},
    "phi":  {"type": "zscore", "agg_axes": None},
}
# Per-trajectory scalars only (timestep is per-snapshot, not a dataset cond key).
gs_cond_keys = [k for k in sorted(gs_cfg.model.conditioning) if k != "timestep"]

gs_valset = CycloneAEDataset(
    split="val", trajectories=ID_VAL,
    backend=KvikIOBackend(0, use_kvikio=False),
    active_keys=list(gs_cfg.dataset.active_keys),
    fields_to_load=["df", "phi", "flux"], probe_targets=[],
    path=DATA_PATH, random_seed=int(gs_cfg.seed),
    normalization=gs_norm,
    normalization_scope="dataset",
    spatial_ifft=bool(gs_cfg.dataset.spatial_ifft),
    bundle_seq_length=1,
    offset=int(gs_cfg.dataset.offset),
    separate_zf=bool(gs_cfg.dataset.separate_zf),
    num_workers=4,
    real_potens=bool(gs_cfg.dataset.real_potens),
    decouple_mu=bool(gs_cfg.dataset.get("norm_decouple_mu", False)),
    rank=0,
    conditions=gs_cond_keys,
    normalization_stats=gs_stats,
)
print(f"GyroSwin valset: {len(gs_valset)} samples (gs-bundled normalization)")

In [ ]:
from neugk.plot_utils import plot_nd

flat_idx_map = {(f, t): idx for idx, (f, t) in gs_valset.flat_index_to_file_and_tstep.items()}
fi = 0
meta = gs_valset.metadata[fi]
t_idx = 30  # input timestep; GyroSwin predicts df at t_idx + 1.
sample_in   = gs_valset[flat_idx_map[(fi, t_idx)]]
sample_next = gs_valset[flat_idx_map[(fi, t_idx + 1)]]
df_in = sample_in.df.unsqueeze(0).to(runner.device)

_cond_meta_map = {"itg": "ion_temp_grad", "dg": "density_grad"}
_gs_cond_keys = sorted(list(gs_cfg.model.conditioning))
_nontime = [k for k in _gs_cond_keys if k != "timestep"]
cond_kwargs = {
    k: torch.tensor(
        [float(np.squeeze(meta[_cond_meta_map.get(k, k)]))],
        dtype=torch.float32, device=runner.device,
    )
    for k in _nontime
}
if "timestep" in _gs_cond_keys:
    ts_val = float(meta["timesteps"][t_idx + gs_valset.offsets[fi]])
    cond_kwargs["timestep"] = torch.tensor([ts_val], dtype=torch.float32, device=runner.device)

fh = getattr(gyroswin_model, "flux_head", None)
if fh is not None and hasattr(fh, "condition_keys") and "timestep" not in fh.condition_keys:
    fh.condition_keys = sorted(list(fh.condition_keys) + ["timestep"])

gyroswin_model.eval()
with torch.no_grad():
    out = gyroswin_model(df_in, **cond_kwargs)
pred_df = out["df"][0].cpu()
gt_df_in   = sample_in.df.cpu()    # input @ t
gt_df_next = sample_next.df.cpu()  # target @ t+1 (autoregressive)

rel_l2_next = float((gt_df_next - pred_df).norm() / (gt_df_next.norm() + 1e-12))
rel_l2_same = float((gt_df_in   - pred_df).norm() / (gt_df_in.norm()   + 1e-12))
print(f"input shape: {gt_df_in.shape}, output shape: {pred_df.shape}")
print(f"recon rel err vs t+1 (correct AR target): {rel_l2_next:.4f}")
print(f"recon rel err vs t   (input, sanity):     {rel_l2_same:.4f}")
if rel_l2_next > 0.5:
    print("  !! WARNING rel L2 vs t+1 > 0.5 — weights may not have loaded or "
            "input normalization doesn't match what GyroSwin expects.")
fig = plot_nd(gt_df_next, pred_df, to_wandb=False)
fig.suptitle(f"GyroSwin: target df(t+1) (left) vs predicted (right) — rel L2 = {rel_l2_next:.3f}",
                fontsize=11, y=1.01)


In [ ]:
valset = gs_valset
t_start, t_step, n_snap = 80, 5, 32

_cond_meta_map_fid = {"itg": "ion_temp_grad", "dg": "density_grad"}
_gs_nontime = [k for k in gs_cond_keys if k != "timestep"]
flat_idx_map = {(f, t): idx for idx, (f, t) in valset.flat_index_to_file_and_tstep.items()}

import re as _re

all_dfs, all_conds, all_traj_idx, all_snap_labels = [], [], [], []
traj_labels = []

for fi in range(len(valset.files)):
    meta = valset.metadata[fi]
    fpath = valset.files[fi]
    m = _re.search(r"iteration_(\d+)", fpath)
    label = f"iter_{m.group(1)}" if m else f"f{fi}"

    cond_vals = [float(np.squeeze(meta[_cond_meta_map_fid.get(k, k)])) for k in _gs_nontime]

    snap_indices = [t_start + j * t_step for j in range(n_snap)]
    n_ts = len(meta["timesteps"])
    snap_indices = [s for s in snap_indices if s < n_ts]

    count = 0
    for si in snap_indices:
        t_idx = si - valset.offsets[fi]
        if t_idx < 0 or (fi, t_idx) not in flat_idx_map:
            continue
        sample = valset[flat_idx_map[(fi, t_idx)]]
        if sample.df is None:
            continue
        all_dfs.append(sample.df)
        all_conds.append(torch.tensor(cond_vals, dtype=torch.float32))
        all_traj_idx.append(len(traj_labels))
        all_snap_labels.append(f"{label}:{si}")
        count += 1

    if count >= 2:
        traj_labels.append(label)
        print(f"  {label}: {count} snapshots")
    else:
        for _ in range(count):
            all_dfs.pop(); all_conds.pop(); all_traj_idx.pop(); all_snap_labels.pop()

all_traj_idx = np.array(all_traj_idx)

# Extract features for both sources, identical sample batches.
_PER_TRAJ_SOURCES = (
    ("skip_deep",  {"source": "skip",       "decoder_level": -1, "pool": "amax", "cond_keys": _gs_nontime}),
    ("bottleneck", {"source": "bottleneck", "pool": "amax"}),
    ("flux_head",  {"source": "flux_head",  "flux_head_level": 1, "cond_keys": _gs_nontime}),
    ("phi",        {"source": "phi",        "pool": "amax", "cond_keys": _gs_nontime}),
)
features_per_source = {}
for src_name, src_kwargs in _PER_TRAJ_SOURCES:
    feats = []
    for i in tqdm(range(0, len(all_dfs), GEN_BATCH_SIZE),
                   desc=f"extract {src_name}"):
        batch_df = torch.stack(all_dfs[i:i+GEN_BATCH_SIZE])
        batch_cond = torch.stack(all_conds[i:i+GEN_BATCH_SIZE])
        feats.append(extract_gyroswin_latents(
            gyroswin_model, batch_df, device=runner.device,
            condition=batch_cond, **src_kwargs,
        ))
    features_per_source[src_name] = np.concatenate(feats)

n_traj = len(traj_labels)

# Build (FID matrix, cosine matrix) per source.
matrices_per_source = {}
for src_name, all_feats in features_per_source.items():
    n_comp = min(FID_N_COMPONENTS, all_feats.shape[0], all_feats.shape[1])
    pca_fid = PCA(n_components=n_comp).fit(all_feats)
    traj_feats = {ti: all_feats[all_traj_idx == ti] for ti in range(n_traj)}
    traj_feats_pca = {ti: pca_fid.transform(traj_feats[ti]) for ti in range(n_traj)}

    fid_matrix = np.full((n_traj, n_traj), np.nan)
    traj_stats = {ti: compute_statistics(traj_feats_pca[ti]) for ti in range(n_traj)}
    for i in range(n_traj):
        for j in range(i, n_traj):
            v = compute_fid(*traj_stats[i], *traj_stats[j])
            fid_matrix[i, j] = v; fid_matrix[j, i] = v

    feats_norm = all_feats / (np.linalg.norm(all_feats, axis=1, keepdims=True) + 1e-8)
    cos_matrix = feats_norm @ feats_norm.T

    matrices_per_source[src_name] = {
        "fid": fid_matrix, "cos": cos_matrix,
        "raw_dim": all_feats.shape[1], "pca_dim": n_comp,
        "explained_var": float(pca_fid.explained_variance_ratio_.sum()),
    }
    print(f"  {src_name}: PCA {all_feats.shape[1]} -> {n_comp} "
          f"({pca_fid.explained_variance_ratio_.sum():.1%} var)")

# 2x2 plot: row 0 = middle (bottleneck), row 1 = flux_head; col 0 = FID, col 1 = cosine.
traj_boundaries = []
cur = 0
for ti in range(n_traj):
    n = int((all_traj_idx == ti).sum())
    traj_boundaries.append((cur, cur + n, traj_labels[ti]))
    cur += n

fig, axes = plt.subplots(2, 2, figsize=(16, 14))
_row_labels = (("bottleneck", "lowest middle (df_unet bottleneck)"),
               ("flux_head",  "flux_head (physics-targeted pre-MLP)"))

for ri, (src_name, src_title) in enumerate(_row_labels):
    M = matrices_per_source[src_name]
    fid_matrix, cos_matrix = M["fid"], M["cos"]

    vmax = np.nanmax(fid_matrix[~np.eye(n_traj, dtype=bool)]) if n_traj > 1 else 1.0
    im0 = axes[ri, 0].imshow(fid_matrix, cmap="YlOrRd", vmin=0, vmax=vmax)
    axes[ri, 0].set_xticks(range(n_traj))
    axes[ri, 0].set_xticklabels(traj_labels, rotation=45, ha="right", fontsize=9)
    axes[ri, 0].set_yticks(range(n_traj))
    axes[ri, 0].set_yticklabels(traj_labels, fontsize=9)
    for i in range(n_traj):
        for j in range(n_traj):
            if np.isfinite(fid_matrix[i, j]):
                axes[ri, 0].text(j, i, f"{fid_matrix[i,j]:.1f}", ha="center", va="center",
                                 fontsize=9, fontweight="bold",
                                 color="white" if fid_matrix[i, j] > 0.6 * vmax else "black")
    plt.colorbar(im0, ax=axes[ri, 0], label="FID", fraction=0.046)
    axes[ri, 0].set_title(f"{src_title}: pairwise FID per trajectory  "
                           f"(raw {M['raw_dim']} -> pca {M['pca_dim']}, "
                           f"{M['explained_var']:.0%} var)", fontsize=10)

    im1 = axes[ri, 1].imshow(cos_matrix, cmap="RdBu_r", vmin=-1, vmax=1)
    for si, (s1, e1, l1) in enumerate(traj_boundaries):
        axes[ri, 1].axhline(s1 - 0.5, color="k", lw=0.5, alpha=0.5)
        axes[ri, 1].axvline(s1 - 0.5, color="k", lw=0.5, alpha=0.5)
        mid1 = (s1 + e1) / 2
        axes[ri, 1].text(-1.5, mid1, l1, ha="right", va="center", fontsize=7, fontweight="bold")
        axes[ri, 1].text(mid1, -1.5, l1, ha="center", va="bottom", fontsize=7, fontweight="bold", rotation=45)
        for sj, (s2, e2, l2) in enumerate(traj_boundaries):
            block_mean = cos_matrix[s1:e1, s2:e2].mean()
            axes[ri, 1].text((s2 + e2) / 2, mid1, f"{block_mean:.2f}",
                             ha="center", va="center", fontsize=7, fontweight="bold",
                             color="black",
                             bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.7))
    axes[ri, 1].axhline(cur - 0.5, color="k", lw=0.5, alpha=0.5)
    axes[ri, 1].axvline(cur - 0.5, color="k", lw=0.5, alpha=0.5)
    axes[ri, 1].set_xticks([]); axes[ri, 1].set_yticks([])
    plt.colorbar(im1, ax=axes[ri, 1], label="cosine sim", fraction=0.046)
    axes[ri, 1].set_title(f"{src_title}: pairwise cosine similarity per snapshot",
                           fontsize=10)

fig.tight_layout()

In [ ]:
runner.model.eval()

# Real samples: gs-normalized via gs_valset.
n = max(1, int(FID_FRAC * len(gs_valset)))
real_s = [gs_valset[i].df for i in tqdm(range(n), desc="Loading real (gs-norm)")]

_cond_meta_map = {"itg": "ion_temp_grad", "dg": "density_grad"}
_gs_nontime = [k for k in gs_cond_keys if k != "timestep"]
real_conds_list, real_fis = [], []
for idx in range(n):
    fi, _ = gs_valset.flat_index_to_file_and_tstep[idx]
    meta = gs_valset.metadata[fi]
    real_conds_list.append(torch.tensor(
        [float(np.squeeze(meta[_cond_meta_map.get(k, k)])) for k in _gs_nontime],
        dtype=torch.float32,
    ))
    real_fis.append(fi)

_diff_cond_keys = sorted(runner.cfg.model.conditioning)
diff_conds = []
for idx in range(n):
    fi, _ = gs_valset.flat_index_to_file_and_tstep[idx]
    meta = gs_valset.metadata[fi]
    diff_conds.append(torch.tensor(
        [float(np.squeeze(meta[_cond_meta_map.get(k, k)])) for k in _diff_cond_keys],
        dtype=torch.float32,
    ))
diff_conds = torch.stack(diff_conds)

# Gen samples: diffusion → denormalize via diff valset → re-normalize with gs_stats.
_gs_df_full = gs_stats["df"]["full"]
_gs_mean = torch.as_tensor(np.asarray(_gs_df_full["mean"]), dtype=torch.float32)
_gs_std  = torch.as_tensor(np.asarray(_gs_df_full["std"]),  dtype=torch.float32)
gen_s, gen_conds_list = [], []
for i in tqdm(range(0, n, GEN_BATCH_SIZE), desc="Generating (diff -> gs-norm)"):
    c = diff_conds[i : i + GEN_BATCH_SIZE].to(runner.device)
    with torch.no_grad():
        p = runner.sample(c, steps=N_DENOISING_STEPS, latent_only=False)
    for j in range(p["df"].shape[0]):
        df_diff_norm = p["df"][j].cpu()
        fi = real_fis[i + j]
        df_raw = runner.valsets[0].denormalize(fi, df=df_diff_norm)
        mean_b = _gs_mean.view(_gs_mean.shape + (1,) * (df_raw.ndim - _gs_mean.ndim))
        std_b  = _gs_std.view(_gs_std.shape  + (1,) * (df_raw.ndim - _gs_std.ndim))
        gen_s.append(((df_raw - mean_b) / std_b).to(df_diff_norm.dtype))
        gen_conds_list.append(real_conds_list[i + j])

print("Extracting GyroSwin features per latent source ...")
gs_features = {}
_SOURCES = (
    ("skip_deep",  {"source": "skip",       "decoder_level": -1, "pool": "amax", "cond_keys": _gs_nontime}),
    ("bottleneck", {"source": "bottleneck", "pool": "amax"}),
    ("flux_head",  {"source": "flux_head",  "flux_head_level": 1, "cond_keys": _gs_nontime}),
    ("phi",        {"source": "phi",        "pool": "amax", "cond_keys": _gs_nontime}),
)
for src_name, src_kwargs in _SOURCES:
    real_feats, gen_feats = [], []
    for i in tqdm(range(0, n, GEN_BATCH_SIZE), desc=f"feats:{src_name}"):
        r_batch = torch.stack(real_s[i:i+GEN_BATCH_SIZE])
        g_batch = torch.stack(gen_s[i:i+GEN_BATCH_SIZE])
        r_cond  = torch.stack(real_conds_list[i:i+GEN_BATCH_SIZE])
        g_cond  = torch.stack(gen_conds_list[i:i+GEN_BATCH_SIZE])
        real_feats.append(extract_gyroswin_latents(
            gyroswin_model, r_batch, device=runner.device, condition=r_cond, **src_kwargs,
        ))
        gen_feats.append(extract_gyroswin_latents(
            gyroswin_model, g_batch, device=runner.device, condition=g_cond, **src_kwargs,
        ))
    Xr = np.concatenate(real_feats, axis=0)
    Xg = np.concatenate(gen_feats,  axis=0)
    gs_features[src_name] = (Xr, Xg)
    print(f"  {src_name:11s}  raw real {Xr.shape}  raw gen {Xg.shape}")

# Default X_real/X_gen used by downstream cells = bottleneck (raw, pre-PCA).
X_real, X_gen = gs_features["bottleneck"]

print(f"Real: {X_real.shape}, Gen: {X_gen.shape}")

In [ ]:
def _safe_pca_fid(Xr, Xg, n_components):
    """PCA + FID with n_components clamped to min(n_samples, n_features)."""
    n_max = min(Xr.shape[0], Xr.shape[1])
    n_comp = min(n_components, n_max) if n_components else n_max
    if n_comp < Xr.shape[1]:
        _pca = PCA(n_components=n_comp)
        Xr_p = _pca.fit_transform(Xr)
        Xg_p = _pca.transform(Xg)
        ev = float(_pca.explained_variance_ratio_.sum())
    else:
        Xr_p, Xg_p, ev = Xr, Xg, 1.0
    m_r, s_r = compute_statistics(Xr_p)
    m_g, s_g = compute_statistics(Xg_p)
    return compute_fid(m_r, s_r, m_g, s_g), Xr_p.shape[1], ev

# Bottleneck FID (default downstream X_real / X_gen).
fid_global, _pca_dim, _ev = _safe_pca_fid(X_real, X_gen, n_components=128)
print(f"Global {fid_label} (bottleneck): {fid_global:.4f}  "
      f"(raw {X_real.shape[1]} -> pca {_pca_dim}, {_ev:.0%} var)")

# Per-source FID for the GyroSwin path: bottleneck (lowest middle) vs flux_head.
gs_fid_per_source = {}
if gs_features:
    print("\n  per-source FID (PCA-reduced):")
    for src_name, (Xr, Xg) in gs_features.items():
        fid_v, pca_dim, ev = _safe_pca_fid(Xr, Xg, n_components=128)
        gs_fid_per_source[src_name] = fid_v
        print(f"    {src_name:20s}  FID = {fid_v:>14.4f}   "
              f"(raw {Xr.shape[1]} -> pca {pca_dim}, {ev:.0%} var)")


In [ ]:
per_traj_fids = {}

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

mr, mg = X_real.mean(0), X_gen.mean(0)
axes[0, 0].scatter(mr, mg, alpha=0.3, s=10, color="#264653")
lim = [min(mr.min(), mg.min()), max(mr.max(), mg.max())]
axes[0, 0].plot(lim, lim, "r--", alpha=0.8)
axes[0, 0].set(title="feature means", xlabel="real", ylabel="gen")
axes[0, 0].grid(True, alpha=0.15)

sr, sg = X_real.std(0), X_gen.std(0)
axes[0, 1].scatter(sr, sg, alpha=0.3, s=10, color="#e9c46a")
lim = [min(sr.min(), sg.min()), max(sr.max(), sg.max())]
axes[0, 1].plot(lim, lim, "r--", alpha=0.8)
axes[0, 1].set(title="feature stds", xlabel="real", ylabel="gen")
axes[0, 1].grid(True, alpha=0.15)

if per_traj_fids:
    axes[1, 0].hist(list(per_traj_fids.values()), bins=30, color="#2a9d8f", edgecolor="k", alpha=0.8)
    axes[1, 0].axvline(fid_global, color="red", ls="--", lw=1.5, label=f"global={fid_global:.1f}")
    axes[1, 0].set(title=f"per-traj {fid_label}", xlabel="FID")
    axes[1, 0].legend()
else:
    axes[1, 0].text(0.5, 0.5, "N/A", ha="center", va="center", transform=axes[1, 0].transAxes)

real_norm = X_real / (np.linalg.norm(X_real, axis=1, keepdims=True) + 1e-8)
gen_norm = X_gen / (np.linalg.norm(X_gen, axis=1, keepdims=True) + 1e-8)
cos_per_sample = np.sum(real_norm * gen_norm, axis=1)
axes[1, 1].hist(cos_per_sample, bins=40, color="#2a9d8f", edgecolor="k", alpha=0.8)
axes[1, 1].axvline(cos_per_sample.mean(), color="red", ls="--", lw=1.5,
                    label=f"mean={cos_per_sample.mean():.3f}")
axes[1, 1].set(title="cosine similarity (real vs gen, paired)", xlabel="cosine sim")
axes[1, 1].legend(); axes[1, 1].grid(True, alpha=0.15)

fig.suptitle(f"{fid_label} = {fid_global:.2f} | mean cos sim = {cos_per_sample.mean():.3f}", fontweight="bold")
fig.tight_layout()

## 3. Warm Restart Evaluation

In [ ]:
from gyaradax import gk_from_gkw_dir

In [ ]:
from neurips_warm_restarts import run_trajectory

N_EVAL_STEPS = 3000
N_RESTARTS   = 4    # diffusion samples per traj; metrics on the across-restart mean trace
SCALE_RATIO_FAIL = 5.0
WARM_RESTART_TRAJS = [f.replace(".h5", "") for f in ID_VAL + OOD_VAL]
GT_SUBSAMPLE = 1
N_GT_TAIL = 3 * 80
RAW_KYSPEC_TO_INTEGRATOR = 16.0

warm_results = {}
_cond_keys = sorted(runner.cfg.model.conditioning)


def _avg_logs(logs):
    """Average warm-trajectory logs across restarts. Time array is taken from
    the first restart (deterministic given fixed solver). Per-step stacks
    along axis 0 then mean over restarts."""
    keys = [k for k in logs[0] if k != "df_final"]
    out = {}
    for k in keys:
        arrays = [np.asarray(l[k]) for l in logs]
        if k == "time":
            out[k] = arrays[0]
        else:
            out[k] = np.mean(np.stack(arrays, axis=0), axis=0)
    out["n_restarts"] = len(logs)
    return out


for traj in WARM_RESTART_TRAJS:
    print(f"\n{'=' * 88}\nTrajectory {traj}\n{'=' * 88}")
    if traj.startswith("ood_iteration_"):
        _gkw_path = os.path.join(GKW_RAW_DIR, "ood", traj[len("ood_"):])
    else:
        _gkw_path = os.path.join(GKW_RAW_DIR, traj)
    df_gt, geometry, params, state_init, pre = gk_from_gkw_dir(
        _gkw_path, mixed_precision=True, k_index=80,
    )
    _param_map = {"itg": "rlt", "dg": "rln", "s_hat": "shat", "q": "q"}
    cond = torch.tensor(
        [[float(getattr(params, _param_map[k])) for k in _cond_keys]],
        dtype=torch.float32, device=runner.device,
    )
    cond_b = cond.expand(N_RESTARTS, -1).contiguous()

    # Sample N_RESTARTS independent diffusion outputs at this condition.
    with torch.no_grad():
        _decoded_b = runner.sample(cond_b, latent_only=False,
                                     steps=N_DENOISING_STEPS)["df"].cpu()
    fi_in_valset = next((idx for idx, fpath in enumerate(runner.valsets[0].files)
                        if traj in fpath), 0)

    gt_max = float(np.max(np.abs(np.asarray(df_gt))))
    gt_std = float(np.std(np.asarray(df_gt).real)) if np.iscomplexobj(np.asarray(df_gt)) else float(np.std(np.asarray(df_gt)))

    restart_logs = []
    df_preds = []
    for ri in range(N_RESTARTS):
        DF_PRED = runner.valsets[0].denormalize(fi_in_valset, df=_decoded_b[ri]).numpy()
        df_warm = from_model_space(DF_PRED)
        df_preds.append(DF_PRED)

        warm_max = float(np.max(np.abs(np.asarray(df_warm))))
        ratio_max = warm_max / max(gt_max, 1e-30)
        if not np.isfinite(ratio_max) or ratio_max > SCALE_RATIO_FAIL:
            print(f"  restart {ri}: ratio_max={ratio_max:.2f} — SKIPPING this restart")
            continue

        log_i = run_trajectory(
            df_warm, geometry, params, pre, state_init,
            N_EVAL_STEPS, f"{traj}/warm{ri}",
            chunk_size=40, backend="jax", mixed_precision=True,
            print_every=500,
        )
        if not np.all(np.isfinite(log_i["eflux"])):
            print(f"  restart {ri}: NaN in eflux — dropping")
            continue
        restart_logs.append(log_i)
        print(f"  restart {ri}: ok  (Q_final={float(log_i['eflux'][-1]):.3e})")

    if not restart_logs:
        print(f"  !! no usable restarts for {traj}; storing empty log_warm.")
        warm_results[traj] = dict(log_warm={}, log_gt_ref={},
                                   ref_flux_samples=None, df_pred=df_preds[0] if df_preds else None,
                                   split=("OOD" if traj.startswith("ood_") else "ID"))
        continue

    log_warm = _avg_logs(restart_logs)
    print(f"  averaged across {len(restart_logs)} restart(s)")

    ref_flux_samples = None
    log_gt_ref = {}
    fluxes_p = os.path.join(_gkw_path, "fluxes.dat")
    if os.path.isfile(fluxes_p):
        ref_flux_samples = np.asarray(
            np.loadtxt(fluxes_p)[::GT_SUBSAMPLE, 1][-N_GT_TAIL:], dtype=np.float64,
        )
    else:
        print(f"  (no fluxes.dat at {fluxes_p})")

    kyspec_p = os.path.join(_gkw_path, "kyspec")
    if os.path.isfile(kyspec_p):
        log_gt_ref["ky_spec"] = np.loadtxt(kyspec_p)[::GT_SUBSAMPLE][-N_GT_TAIL:] * RAW_KYSPEC_TO_INTEGRATOR
    else:
        print(f"  (no kyspec at {kyspec_p})")

    eflux_p = os.path.join(_gkw_path, "eflux_spectra.dat")
    if os.path.isfile(eflux_p):
        log_gt_ref["fluxspec"] = np.loadtxt(eflux_p)[::GT_SUBSAMPLE][-N_GT_TAIL:]
    else:
        print(f"  (no eflux_spectra.dat at {eflux_p})")

    time_p = os.path.join(_gkw_path, "time.dat")
    if os.path.isfile(time_p):
        log_gt_ref["time"] = np.loadtxt(time_p)[::GT_SUBSAMPLE][-N_GT_TAIL:]

    print(f"  GT raw: flux={None if ref_flux_samples is None else ref_flux_samples.shape}  "
          f"ky_spec={getattr(log_gt_ref.get('ky_spec'), 'shape', None)}  "
          f"fluxspec={getattr(log_gt_ref.get('fluxspec'), 'shape', None)}")

    warm_results[traj] = dict(
        log_warm=log_warm, log_gt_ref=log_gt_ref,
        ref_flux_samples=ref_flux_samples, df_pred=df_preds[0],
        n_restarts=len(restart_logs),
        split=("OOD" if traj.startswith("ood_") else "ID"),
    )
    print(f"  {traj}: stored mean-log over {len(restart_logs)} restart(s).")


In [ ]:
from neurips_warm_restarts import compute_distribution_divergences

WARM_FRAC = 0.95

PRINT_KEYS = (
    "flux_w1", "flux_mmd", "flux_ks_p", "flux_ad", "flux_arima_l2", "flux_r2_hist",
    "ky_spec_w1_mean", "ky_spec_mmd_mean", "ky_spec_ks_p_mean",
    "ky_spec_ad_mean", "ky_spec_arima_l2_mean", "ky_spec_r2_meanlog",
    "fluxspec_w1_mean", "fluxspec_mmd_mean", "fluxspec_ks_p_mean",
    "fluxspec_ad_mean", "fluxspec_arima_l2_mean", "fluxspec_r2_meanlog",
)

# diagnostic: warm raw length, GT length
print(f"{'traj':<22}  {'warm_T':>7}  {'gt_T':>5}")
for traj, res in warm_results.items():
    lw = res.get("log_warm", {})
    if "eflux" not in lw:
        continue
    gt = res.get("ref_flux_samples")
    print(f"{traj:<22}  {len(lw['eflux']):>7}  {len(gt) if gt is not None else 0:>5}")

for traj, res in warm_results.items():
    log_warm = res.get("log_warm")
    if not log_warm or "eflux" not in log_warm:
        res["div_warm"] = {}
        continue
    res["div_warm"] = compute_distribution_divergences(
        log_warm,
        res.get("log_gt_ref", {}),
        ref_flux_samples=res.get("ref_flux_samples"),
        warm_frac=WARM_FRAC,
    )
    print(f"\n  {traj} divergences (warm vs gt saturated):  6 metrics x 3 quantities")
    print(f"  {'metric':32s}  {'value':>14s}")
    for k in PRINT_KEYS:
        v = res["div_warm"].get(k)
        if v is not None and np.isfinite(v):
            print(f"  {k:32s}  {v:>14.4g}")

In [ ]:
n = len(warm_results)
fig, axes = plt.subplots(n, 4, figsize=(22, 4 * n), squeeze=False)
snap_colors = ["#2a9d8f", "#e76f51", "#264653"]

for row, (it, res) in enumerate(warm_results.items()):
    lw = res["log_warm"]
    t = lw["time"]
    ref_flux = res["ref_flux_samples"]
    log_gt_ref = res["log_gt_ref"]

    n_t = len(t)
    snap_idx = [min(5, n_t - 1), n_t // 2, n_t - 1]

    # col 0: ky-spectrum: 3 warm snapshots + GT-saturated mean
    for j, si in enumerate(snap_idx):
        ky_w = np.log10(np.maximum(lw["ky_spec"][si], 1e-30))
        axes[row, 0].plot(ky_w, "-", color=snap_colors[j], lw=1.2, alpha=0.9,
                           label=f"warm t={float(t[si]):.1f}")
    if "ky_spec" in log_gt_ref:
        ky_gt_mean = np.log10(np.maximum(np.asarray(log_gt_ref["ky_spec"]).mean(0), 1e-30))
        axes[row, 0].plot(ky_gt_mean, "k--", lw=1.4, alpha=0.85, label="GT mean")
    axes[row, 0].set(title=f"iter {it}: $W(k_y)$", xlabel="$k_y$ mode",
                       ylabel="log$_{10}$ W")
    axes[row, 0].legend(fontsize=7); axes[row, 0].grid(True, alpha=0.15)

    # col 1: fluxspec in linear (signed) — kyspec is log
    if "fluxspec" in lw:
        for j, si in enumerate(snap_idx):
            fs_w = np.asarray(lw["fluxspec"][si])
            axes[row, 1].plot(fs_w, "-", color=snap_colors[j], lw=1.2, alpha=0.9,
                               label=f"warm t={float(t[si]):.1f}")
        if "fluxspec" in log_gt_ref:
            fs_gt_mean = np.asarray(log_gt_ref["fluxspec"]).mean(0)
            axes[row, 1].plot(fs_gt_mean, "k--", lw=1.4, alpha=0.85, label="GT mean")
        axes[row, 1].axhline(0, color="gray", lw=0.5, alpha=0.4)
        axes[row, 1].set(title=f"iter {it}: $Q(k_y)$", xlabel="$k_y$ mode",
                           ylabel="$Q$")
        axes[row, 1].legend(fontsize=7); axes[row, 1].grid(True, alpha=0.15)
    else:
        axes[row, 1].text(0.5, 0.5, "no fluxspec", ha="center", va="center",
                            transform=axes[row, 1].transAxes); axes[row, 1].set_axis_off()

    # col 2: flux trace + GT reference band
    axes[row, 2].plot(t, lw["eflux"], lw=1, color="#2a9d8f", label="warm")
    if ref_flux is not None and len(ref_flux) > 1:
        m, s = float(np.mean(ref_flux)), float(np.std(ref_flux))
        axes[row, 2].axhspan(m - s, m + s, color="k", alpha=0.08, label="GT ±1σ")
        axes[row, 2].axhline(m, color="k", ls=":", lw=0.8)
    axes[row, 2].set_title(f"iter {it}: flux"); axes[row, 2].legend(fontsize=7)
    axes[row, 2].grid(True, alpha=0.15)

    # col 3: Pearson(log ky_warm, log ky_GT_mean) over time
    if "ky_spec" in log_gt_ref:
        ky_gt_mean = np.log10(np.maximum(np.asarray(log_gt_ref["ky_spec"]).mean(0), 1e-30))
        n_compare = len(lw["ky_spec"])
        r_ts = []
        for i in range(n_compare):
            ky_w = np.log10(np.maximum(lw["ky_spec"][i], 1e-30))
            r_ts.append(pearsonr(ky_w, ky_gt_mean)[0] if len(ky_gt_mean) > 1 else 0.0)
        axes[row, 3].plot(t[:n_compare], r_ts, lw=1, label="Pearson")
        axes[row, 3].axhline(0.95, color="gray", ls="--", lw=0.8)
        axes[row, 3].set_title(f"iter {it}: Pearson($k_y$ warm, GT mean)")
        axes[row, 3].set_ylim(-0.1, 1.05)
        axes[row, 3].legend(fontsize=7); axes[row, 3].grid(True, alpha=0.15)
    else:
        axes[row, 3].text(0.5, 0.5, "no kyspec ref", ha="center", va="center",
                            transform=axes[row, 3].transAxes); axes[row, 3].set_axis_off()

for ax in axes[-1]:
    ax.set_xlabel(r"time $[v_{th}/R]$")
fig.tight_layout()

In [ ]:
SPEC_KEYS = ("ky_spec", "fluxspec")

for it, res in warm_results.items():
    lw = res["log_warm"]
    log_gt_ref = res["log_gt_ref"]
    avail = [s for s in SPEC_KEYS if s in lw and (s == "kx_spec" or s in log_gt_ref)]
    avail_warm_only = [s for s in SPEC_KEYS if s in lw and s not in log_gt_ref]
    keys = [s for s in SPEC_KEYS if s in lw]
    if not keys:
        continue
    n_rows = len(keys)
    fig, axes = plt.subplots(n_rows, 2,
                              figsize=(8.4, 3.2 * n_rows),
                              squeeze=False, sharex="col")
    t = np.asarray(lw["time"])
    for ri, spec_key in enumerate(keys):
        Sw = np.asarray(lw.get(spec_key, []))
        Sr = np.asarray(log_gt_ref.get(spec_key, []))
        # Shared colour limits across both panels for this spectrum.
        stacks = []
        if Sw.ndim == 2 and Sw.size:
            stacks.append(Sw)
        if Sr.ndim == 2 and Sr.size:
            stacks.append(Sr)
        if not stacks:
            continue
        vmin = min(s.min() for s in stacks)
        vmax = max(s.max() for s in stacks)

        # warm panel
        ax = axes[ri, 0]
        if Sw.ndim == 2 and Sw.size:
            T_w, K = Sw.shape
            tt = t[:T_w]
            ax.imshow(Sw,
                      origin="lower", aspect="auto", cmap="magma",
                      extent=[0, K - 1, float(tt[0]), float(tt[-1])],
                      vmin=vmin, vmax=vmax)
        ax.set_title(f"iter {it} — {spec_key} (warm)", fontsize=9)
        ax.set_ylabel(r"time $[v_{th}/R]$")
        ax.set_xlabel(f"{spec_key.split('_')[0]} mode" if spec_key != "fluxspec" else "mode")

        # GT-saturated reference panel
        ax = axes[ri, 1]
        if Sr.ndim == 2 and Sr.size:
            T_r, K = Sr.shape
            im = ax.imshow(Sr,
                           origin="lower", aspect="auto", cmap="magma",
                           extent=[0, K - 1, 0, T_r],
                           vmin=vmin, vmax=vmax)
            cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.02)
            cbar.set_label(r"$\log_{10}$(magnitude)", fontsize=8)
            ax.set_ylabel(f"GT saturated step (last {T_r})")
        else:
            ax.text(0.5, 0.5, "no GT reference", ha="center", va="center",
                    transform=ax.transAxes); ax.set_axis_off()
        ax.set_title(f"iter {it} — {spec_key} (GT saturated)", fontsize=9)
    fig.tight_layout()

## 4. Distributional divergences: warm vs GT-saturated state

The **warm trajectory is short** (∼1k solver steps) and may sit anywhere
on the attractor — it does *not* track the long GT trajectory pointwise.
We therefore compare the warm run's stationary window (default last 50% of
the run) against the **GT saturated-state distribution**:

* **flux**: last 240 rows of `fluxes.dat` (column 2; loaded by
  `load_reference_flux_samples`, equivalent to the last 80 snapshots since
  fluxes.dat is sampled 3× more frequently than the snapshot grid).
* **spectra (kyspec, fluxspec)**: last `N_GT_TAIL_SPEC=80` snapshots from
  the preprocessed `metadata.pkl` of `iteration_{i}_ifft_realpotens`.

The driver `compute_distribution_divergences` runs every paper-spec test —
KS, AD, Wasserstein, Gelman–Rubin R̂, autocorrelation/structure-fn — plus
Wilcoxon (Mann–Whitney rank-sum + signed-rank), MMD-RBF, energy distance,
mean-log-spectrum Pearson/L2, time-averaged-spectrum KL, and the paper's
τ_Q first-passage times. All variants are scored *only on the warm run*;
TTC has been removed.

In [ ]:
# Warm-vs-GT divergences across iterations: rows = (iter, metric), cols = value.
rows = []
for it, res in warm_results.items():
    for k, v in res["div_warm"].items():
        if k.endswith("_error"):
            continue
        rows.append({"iter": it, "metric": k, "value": v})
divergence_long = pd.DataFrame(rows)
divergence_pivot = divergence_long.pivot(index="metric", columns="iter", values="value")
divergence_pivot["mean"]   = divergence_pivot.mean(axis=1)
divergence_pivot["median"] = divergence_pivot.median(axis=1)
divergence_pivot["std"]    = divergence_pivot.std(axis=1)
divergence_pivot = divergence_pivot.sort_values("mean", ascending=True)
print("Warm-run divergences vs GT saturated state (one column per iter, sorted by mean):")
print(divergence_pivot.to_string(float_format=lambda x: f"{x:.4g}"))
divergence_pivot

In [ ]:
# Per-iteration bar plot: one bar per (iter, metric) — no cold companion.
def _metric_family(name):
    if name.startswith("tau_q_"):    return "tau_Q (first-passage)"
    if name.startswith("flux_"):     return "flux scalar"
    if name.startswith("ky_spec_"):  return "ky_spec"
    if name.startswith("kx_spec_"):  return "kx_spec"
    if name.startswith("fluxspec_"): return "fluxspec"
    return "other"
divergence_long["family"] = divergence_long["metric"].map(_metric_family)
families = ["flux scalar", "tau_Q (first-passage)", "ky_spec", "kx_spec", "fluxspec"]
families = [f for f in families if f in set(divergence_long["family"])]

iters = sorted(warm_results.keys())
ITER_COLORS = plt.get_cmap("viridis")(np.linspace(0.15, 0.9, len(iters)))

n_panels = len(families)
fig, axes = plt.subplots(n_panels, 1,
                          figsize=(max(10, len(divergence_long["metric"].unique()) * 0.45),
                                   3.6 * n_panels),
                          squeeze=False)
for pi, family in enumerate(families):
    ax = axes[pi, 0]
    sub = divergence_long[divergence_long["family"] == family]
    metric_names = sorted(sub["metric"].unique())
    n_iters = len(iters)
    bar_w = 0.8 / max(1, n_iters)
    x_centers = np.arange(len(metric_names))
    for ii, it in enumerate(iters):
        vals = []
        for m in metric_names:
            row = sub[(sub["iter"] == it) & (sub["metric"] == m)]
            vals.append(float(row["value"].iloc[0]) if not row.empty else np.nan)
        offset = (ii - (n_iters - 1) / 2.0) * bar_w
        ax.bar(x_centers + offset, vals, bar_w,
               color=ITER_COLORS[ii], alpha=0.9,
               edgecolor="black", linewidth=0.4,
               label=f"iter {it}")
    ax.set_xticks(x_centers)
    ax.set_xticklabels(metric_names, rotation=45, ha="right", fontsize=8)
    ax.set_title(family, fontsize=11)
    ax.grid(True, alpha=0.15, axis="y")
    finite = sub["value"].dropna()
    if (finite > 0).all() and finite.size and finite.max() / max(finite.min(), 1e-30) > 100:
        ax.set_yscale("log")
    if pi == 0:
        ax.legend(fontsize=8, ncol=min(4, len(iters)), loc="upper right")
fig.tight_layout()

In [ ]:
PRUNE_STD_THRESHOLD = 1e-4

M_full = divergence_pivot.drop(columns=[c for c in ("mean", "median", "std")
                                          if c in divergence_pivot.columns])
all_metrics = list(M_full.index)
traj_names  = list(M_full.columns)

FAMILIES    = ("flux", "ky_spec", "fluxspec")
FAMILY_CMAP = {
    "flux":     "Reds",
    "ky_spec":  "Blues",
    "fluxspec": "Oranges",
}
INTRA_CMAP = "RdBu_r"

def _family(m):
    for f in FAMILIES:
        if m.startswith(f + "_"): return f
    return "other"

def _type(m, f):
    return m[len(f) + 1:] if m.startswith(f + "_") else m

# ---- prune NaN / near-constant metrics across trajs --------------------
kept, pruned = [], []
for m in all_metrics:
    v = M_full.loc[m].to_numpy(dtype=float)
    finite = v[np.isfinite(v)]
    if finite.size < 2:
        pruned.append((m, "few finite")); continue
    if finite.std() <= PRUNE_STD_THRESHOLD * (abs(finite.mean()) + 1e-30):
        pruned.append((m, f"near-constant std={finite.std():.2e}")); continue
    kept.append(m)
if pruned:
    print(f"pruned {len(pruned)}/{len(all_metrics)} metrics:")
    for m, reason in pruned[:30]:
        print(f"  - {m}: {reason}")
    if len(pruned) > 30:
        print(f"  … and {len(pruned) - 30} more")

# ---- group ---------------------------------------------------------------
families_metrics = {f: [m for m in kept if _family(m) == f] for f in FAMILIES}
families_metrics = {f: ms for f, ms in families_metrics.items() if ms}

# ---- per-family figure: (left) metric × traj heatmap (row-normalised),
#                         (right) metric × metric Pearson r (cluster spotting)
for fam, ms in families_metrics.items():
    cmap_traj = FAMILY_CMAP.get(fam, "Greys")
    short = [_type(m, fam) for m in ms]

    # Panel 1: metric × traj, row-normalised so each metric is comparable.
    M = M_full.loc[ms].to_numpy(dtype=float)
    M_norm = np.zeros_like(M)
    for i in range(M.shape[0]):
        finite = M[i][np.isfinite(M[i])]
        if finite.size == 0:
            continue
        lo, hi = finite.min(), finite.max()
        rng = max(hi - lo, 1e-12)
        M_norm[i, :] = (M[i, :] - lo) / rng

    # Panel 2: intra-family metric × metric Pearson r across trajs.
    K = len(ms)
    R = np.full((K, K), np.nan)
    for i in range(K):
        for j in range(K):
            xi = M[i]; xj = M[j]
            mask = np.isfinite(xi) & np.isfinite(xj)
            if mask.sum() < 3 or np.std(xi[mask]) == 0 or np.std(xj[mask]) == 0:
                continue
            R[i, j] = float(np.corrcoef(xi[mask], xj[mask])[0, 1])

    fig, axes = plt.subplots(
        1, 2,
        figsize=(max(8, 0.45 * (len(traj_names) + K)), max(3, 0.34 * K)),
        gridspec_kw={"width_ratios": [max(1, len(traj_names)), max(1, K)],
                     "wspace": 0.35},
    )

    # left: metric × traj
    im0 = axes[0].imshow(M_norm, cmap=cmap_traj, aspect="auto", vmin=0, vmax=1)
    axes[0].set_yticks(range(K)); axes[0].set_yticklabels(short, fontsize=8)
    axes[0].set_xticks(range(len(traj_names)))
    axes[0].set_xticklabels([str(t) for t in traj_names], rotation=30, ha="right", fontsize=8)
    for i in range(K):
        for j in range(len(traj_names)):
            v = M[i, j]
            if np.isfinite(v):
                axes[0].text(j, i, f"{v:.2g}", ha="center", va="center",
                             fontsize=6,
                             color="black" if 0.25 < M_norm[i, j] < 0.75 else "white")
    axes[0].set_title(f"{fam}: divergence per (metric, traj) — row-normalised",
                       fontsize=10, fontweight="bold")
    plt.colorbar(im0, ax=axes[0], fraction=0.04, pad=0.02, label="row-normalised")

    # right: metric × metric Pearson r
    im1 = axes[1].imshow(R, cmap=INTRA_CMAP, aspect="equal", vmin=-1, vmax=1)
    axes[1].set_xticks(range(K)); axes[1].set_xticklabels(short, rotation=45, ha="right", fontsize=8)
    axes[1].set_yticks(range(K)); axes[1].set_yticklabels(short, fontsize=8)
    for i in range(K):
        for j in range(K):
            v = R[i, j]
            if np.isfinite(v):
                axes[1].text(j, i, f"{v:+.2f}", ha="center", va="center",
                             fontsize=6,
                             color="white" if abs(v) > 0.5 else "black")
    axes[1].set_title(f"{fam}: metric × metric Pearson r  "
                       "(blocks of red ⇒ redundant cluster)",
                       fontsize=10, fontweight="bold")
    plt.colorbar(im1, ax=axes[1], fraction=0.04, pad=0.02, label="Pearson r")

    fig.suptitle(f"family: {fam}   ({K} metrics across {len(traj_names)} trajs)",
                 fontsize=12, fontweight="bold", y=1.0)
    fig.tight_layout()

if gs_fid_per_source:
    print("\nGlobal GyroSwin FID per source (constant across trajs):")
    for src, v in gs_fid_per_source.items():
        print(f"  {src:20s}  FID = {v:.4f}")


In [ ]:
import pickle as _pickle
from neugk.utils import separate_zf as _sep_zf

K_GEN = 16
N_GT_SAMPLES = 16

LATENT_LEVELS = {
    "skip_deep":  {"source": "skip",       "decoder_level": -1, "pool": "amax"},
    "bottleneck": {"source": "bottleneck", "pool": "amax"},
    "flux_head":  {"source": "flux_head",  "flux_head_level": 1},
    "phi":        {"source": "phi",        "pool": "amax"},
}

_gs_mean_t = torch.as_tensor(np.asarray(gs_stats["df"]["full"]["mean"]), dtype=torch.float32)
_gs_std_t  = torch.as_tensor(np.asarray(gs_stats["df"]["full"]["std"]),  dtype=torch.float32)
_gs_nontime_local = [k for k in gs_cond_keys if k != "timestep"]
_diff_cond_keys   = sorted(runner.cfg.model.conditioning)
_meta_map_local   = {"itg": "ion_temp_grad", "dg": "density_grad"}


def _gs_norm(df_model):
    m = _gs_mean_t.view(_gs_mean_t.shape + (1,) * (df_model.ndim - _gs_mean_t.ndim))
    s = _gs_std_t.view(_gs_std_t.shape  + (1,) * (df_model.ndim - _gs_std_t.ndim))
    return (df_model - m) / s


def _build_gt_latents_id(traj, levels, n_samples=N_GT_SAMPLES):
    """Last `n_samples` preprocessed bins → GyroSwin features per level.
    ID-only (OOD has no preprocessed bins on disk)."""
    gt_dir = os.path.join(DATA_PATH, f"{traj}_ifft_realpotens")
    meta_p = os.path.join(gt_dir, "metadata.pkl")
    if not os.path.isfile(meta_p):
        return None
    with open(meta_p, "rb") as f:
        meta = _pickle.load(f)
    n_ts = len(meta["timesteps"]); res = meta["resolution"]
    cond_vals = torch.tensor(
        [[float(np.squeeze(meta[_meta_map_local.get(k, k)])) for k in _gs_nontime_local]],
        dtype=torch.float32,
    )
    sep_zf = bool(runner.cfg.dataset.separate_zf)
    out = {name: [] for name in levels}
    for t in range(max(0, n_ts - n_samples), n_ts):
        bin_path = os.path.join(gt_dir, "data", f"timestep_{t:05d}.bin")
        if not os.path.isfile(bin_path):
            continue
        arr = np.fromfile(bin_path, dtype=np.float32).reshape(2, *res)
        if sep_zf:
            arr = _sep_zf(arr, dim=0)
        df_t = _gs_norm(torch.as_tensor(arr, dtype=torch.float32)).unsqueeze(0).to(runner.device)
        for name, kwargs in levels.items():
            kw = dict(kwargs)
            if kw["source"] in ("flux_head", "decoder", "skip", "phi"):
                kw["cond_keys"] = _gs_nontime_local
            try:
                feat = extract_gyroswin_latents(
                    gyroswin_model, df_t, device=runner.device,
                    condition=cond_vals, **kw,
                )[0]
            except Exception:
                feat = np.zeros(1, dtype=np.float32)
            out[name].append(feat)
    return {name: np.stack(v) for name, v in out.items() if v}


def _generate_traj_samples_id(traj, k_gen=K_GEN):
    """Draw `k_gen` diffusion samples for `traj`'s condition vector and
    return them in gs-normalised space, ready for the GyroSwin extractor."""
    fi = next((idx for idx, fpath in enumerate(runner.valsets[0].files)
                if traj in fpath), None)
    if fi is None:
        return None, None
    meta_diff = runner.valsets[0].metadata[fi]
    diff_cond = torch.tensor(
        [float(np.squeeze(meta_diff[_meta_map_local.get(k, k)])) for k in _diff_cond_keys],
        dtype=torch.float32,
    )
    gs_cond = torch.tensor(
        [float(np.squeeze(meta_diff[_meta_map_local.get(k, k)])) for k in _gs_nontime_local],
        dtype=torch.float32,
    )
    runner.model.eval()
    gen_dfs = []
    n_done = 0
    while n_done < k_gen:
        b = min(GEN_BATCH_SIZE, k_gen - n_done)
        c = diff_cond.unsqueeze(0).expand(b, -1).contiguous().to(runner.device)
        with torch.no_grad():
            out = runner.sample(c, steps=N_DENOISING_STEPS, latent_only=False)
        for j in range(out["df"].shape[0]):
            df_diff_norm = out["df"][j].cpu()
            df_raw = runner.valsets[0].denormalize(fi, df=df_diff_norm)
            gen_dfs.append(_gs_norm(df_raw).to(df_diff_norm.dtype))
        n_done += b
    return gen_dfs, gs_cond


def _gen_features(gen_dfs, gs_cond, levels):
    feats_per_level = {name: [] for name in levels}
    cond_batch_template = gs_cond.unsqueeze(0)
    for i in range(0, len(gen_dfs), GEN_BATCH_SIZE):
        batch_df = torch.stack(gen_dfs[i:i + GEN_BATCH_SIZE])
        b = batch_df.shape[0]
        cond_batch = cond_batch_template.expand(b, -1).contiguous()
        for name, kwargs in levels.items():
            kw = dict(kwargs)
            if kw["source"] in ("flux_head", "decoder", "skip", "phi"):
                kw["cond_keys"] = _gs_nontime_local
            try:
                f = extract_gyroswin_latents(
                    gyroswin_model, batch_df, device=runner.device,
                    condition=cond_batch, **kw,
                )
            except Exception:
                f = np.zeros((b, 1), dtype=np.float32)
            feats_per_level[name].append(f)
    return {name: np.concatenate(v, axis=0) for name, v in feats_per_level.items() if v}


trajs = list(warm_results.keys())
levels = list(LATENT_LEVELS.keys())
per_iter_fids = {}

for traj in trajs:
    if traj.startswith("ood_"):
        print(f"  {traj}: OOD has no preprocessed bins; skipping per-traj FID.")
        continue
    gt_per_level = _build_gt_latents_id(traj, LATENT_LEVELS)
    if gt_per_level is None:
        print(f"  {traj}: no preprocessed metadata; skipping.")
        continue
    gen_dfs, gs_cond = _generate_traj_samples_id(traj)
    if gen_dfs is None:
        print(f"  {traj}: no valset match; skipping.")
        continue
    gen_per_level = _gen_features(gen_dfs, gs_cond, LATENT_LEVELS)

    iter_fids = {}
    for lv in levels:
        if lv not in gen_per_level or lv not in gt_per_level:
            iter_fids[lv] = float("nan"); continue
        A = gen_per_level[lv]; B = gt_per_level[lv]
        if A.ndim != 2 or B.ndim != 2 or A.shape[1] != B.shape[1]:
            iter_fids[lv] = float("nan"); continue
        n_comp = max(2, min(A.shape[1], min(A.shape[0], B.shape[0]) - 1))
        try:
            pooled = np.concatenate([A, B], axis=0)
            pca = PCA(n_components=min(n_comp, pooled.shape[1])).fit(pooled)
            Ap = pca.transform(A); Bp = pca.transform(B)
        except Exception:
            Ap, Bp = A, B
        mu_g, sig_g = compute_statistics(Ap)
        mu_r, sig_r = compute_statistics(Bp)
        iter_fids[lv] = float(compute_fid(mu_g, sig_g, mu_r, sig_r))
    per_iter_fids[traj] = iter_fids
    print(f"  {traj}: " + "  ".join(f"FID({lv})={iter_fids[lv]:.3f}" for lv in levels))

trajs_done = list(per_iter_fids.keys())
if trajs_done:
    fig, ax = plt.subplots(figsize=(max(8, 0.6 * len(trajs_done)),
                                     max(2.5, 0.45 * len(levels))))
    M = np.array([[per_iter_fids[t].get(lv, np.nan) for t in trajs_done]
                   for lv in levels], dtype=float)
    im = ax.imshow(M, cmap="viridis", aspect="auto")
    ax.set_yticks(range(len(levels)));   ax.set_yticklabels(levels, fontsize=10, fontweight="bold")
    ax.set_xticks(range(len(trajs_done))); ax.set_xticklabels(trajs_done, rotation=30, ha="right", fontsize=8)
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            v = M[i, j]
            if np.isfinite(v):
                ax.text(j, i, f"{v:.2g}", ha="center", va="center", fontsize=7,
                        color="white" if v > np.nanmedian(M) else "black")
    ax.set_title("Single-shot generative FID per (U-Net level, trajectory)",
                  fontsize=11, fontweight="bold")
    plt.colorbar(im, ax=ax, fraction=0.04, pad=0.02, label="FID (gen vs GT bins)")
    fig.tight_layout()

    print("\n=== single-shot FID summary (lower = closer to GT distribution) ===")
    print(f"  {'level':>11s}  " + "  ".join(f"{t:>14s}" for t in trajs_done) + f"  {'mean':>10s}")
    for lv in levels:
        vals = np.array([per_iter_fids[t].get(lv, np.nan) for t in trajs_done], dtype=float)
        cells = "  ".join(f"{v:>14.3g}" if np.isfinite(v) else f"{'nan':>14s}" for v in vals)
        print(f"  {lv:>11s}  {cells}  {np.nanmean(vals):>10.3g}")

In [ ]:
_ns = globals()
_pif = _ns.get("per_iter_fids")
_warm = _ns.get("warm_results", {})
if not _pif:
    raise RuntimeError("per_iter_fids is empty or undefined — run the "
                        "`latent_dashboard` cell first.")
per_iter_fids = _pif

# Build per-traj table of metrics + FIDs.
rows = []
for traj, fids in per_iter_fids.items():
    if traj not in warm_results or not warm_results[traj].get("div_warm"):
        continue
    row = {"traj": traj, "split": warm_results[traj].get("split", "ID")}
    for lv, v in fids.items():
        row[f"FID_{lv}"] = v
    for k, v in warm_results[traj]["div_warm"].items():
        if k.endswith("_error"):
            continue
        row[k] = v
    rows.append(row)

if len(rows) < 3:
    print(f"only {len(rows)} trajs with both FID + divergences — Pearson r needs ≥ 3")
else:
    corr_df = pd.DataFrame(rows)
    fid_cols = [c for c in corr_df.columns if c.startswith("FID_")]
    raw_metric_cols = [c for c in corr_df.columns
                       if c not in (["traj", "split"] + fid_cols)]

    FAMILY_ORDER = ("flux", "ky_spec", "fluxspec", "other")
    TYPE_ORDER   = ("w1", "w1_mean", "mmd", "mmd_mean",
                    "ks_p", "ks_p_mean",
                    "ad", "ad_mean",
                    "arima_l2", "arima_l2_mean",
                    "r2_hist", "r2_meanlog")

    def _family(name):
        if name.startswith("flux_"):     return "flux"
        if name.startswith("ky_spec_"):  return "ky_spec"
        if name.startswith("fluxspec_"): return "fluxspec"
        return "other"
    def _type(name, fam):
        if fam == "flux":     return name[len("flux_"):]
        if fam == "ky_spec":  return name[len("ky_spec_"):]
        if fam == "fluxspec": return name[len("fluxspec_"):]
        return name
    def _sort_key(name):
        fam = _family(name); typ = _type(name, fam)
        fam_idx = FAMILY_ORDER.index(fam) if fam in FAMILY_ORDER else len(FAMILY_ORDER)
        typ_idx = TYPE_ORDER.index(typ) if typ in TYPE_ORDER else len(TYPE_ORDER) + hash(typ) % 1000
        return (fam_idx, typ_idx, name)

    metric_cols = sorted(raw_metric_cols, key=_sort_key)
    metric_families = [_family(m) for m in metric_cols]
    fam_label_pos = {}
    for fam in FAMILY_ORDER:
        idxs = [i for i, f in enumerate(metric_families) if f == fam]
        if idxs:
            fam_label_pos[fam] = (idxs[0], idxs[-1])

    corr_matrix = pd.DataFrame(index=metric_cols, columns=fid_cols, dtype=float)
    for m in metric_cols:
        for fcol in fid_cols:
            x = corr_df[m].to_numpy(dtype=float)
            y = corr_df[fcol].to_numpy(dtype=float)
            mask = np.isfinite(x) & np.isfinite(y)
            if mask.sum() < 3 or np.std(x[mask]) == 0 or np.std(y[mask]) == 0:
                corr_matrix.loc[m, fcol] = np.nan
            else:
                corr_matrix.loc[m, fcol] = float(np.corrcoef(x[mask], y[mask])[0, 1])

    families_present = [fam for fam in FAMILY_ORDER if fam in fam_label_pos]
    height_ratios = [(fam_label_pos[f][1] - fam_label_pos[f][0] + 1)
                     for f in families_present]
    fig, axes = plt.subplots(
        len(families_present), 1, sharex=True,
        figsize=(max(6, 1.6 * len(fid_cols)), max(4, 0.34 * len(metric_cols))),
        gridspec_kw={"height_ratios": height_ratios, "hspace": 0.25},
    )
    if len(families_present) == 1:
        axes = [axes]
    for ax_i, fam in zip(axes, families_present):
        lo, hi = fam_label_pos[fam]
        block_metrics = metric_cols[lo:hi + 1]
        block_short = [_type(m, fam) for m in block_metrics]
        Mb = corr_matrix.loc[block_metrics].to_numpy(dtype=float)
        im = ax_i.imshow(Mb, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
        ax_i.set_yticks(range(len(block_short)))
        ax_i.set_yticklabels(block_short, fontsize=8)
        ax_i.set_xticks(range(len(fid_cols)))
        ax_i.set_xticklabels([c.replace("FID_", "FID(") + ")" for c in fid_cols],
                              rotation=20, ha="right", fontsize=8)
        for r in range(Mb.shape[0]):
            for c2 in range(Mb.shape[1]):
                v = Mb[r, c2]
                if np.isfinite(v):
                    ax_i.text(c2, r, f"{v:+.2f}", ha="center", va="center",
                              fontsize=7,
                              color="white" if abs(v) > 0.5 else "black")
        ax_i.set_ylabel(fam, fontsize=11, fontweight="bold", rotation=90, labelpad=8)
        for spine in ax_i.spines.values():
            spine.set_linewidth(1.6); spine.set_edgecolor("#264653")
    fig.suptitle(
        f"Pearson r: 18 trajectory metrics × FID at {len(fid_cols)} U-Net levels",
        fontsize=11, fontweight="bold", y=0.995,
    )
    fig.colorbar(im, ax=axes, fraction=0.06, pad=0.02, label="Pearson r")

    # Leaderboard: rank within family by max |r| across all FID levels.
    print(f"\n=== sorted by family, then by max |r| across {len(fid_cols)} FID levels ===")
    lb = corr_matrix.copy()
    lb["family"] = [_family(m) for m in lb.index]
    lb["best_abs_r"] = lb[fid_cols].abs().max(axis=1)
    lb = lb.sort_values(
        ["family", "best_abs_r"], ascending=[True, False],
        key=lambda col: col.map({f: i for i, f in enumerate(FAMILY_ORDER)})
                          if col.name == "family" else col,
    ).drop(columns="best_abs_r")
    print(lb.to_string(float_format=lambda x: f"{x:+.3f}"))

In [ ]:
R_REDUNDANT = 0.95   # |r| above which two metrics are considered redundant
TOP_N       = 12     # leaderboard length per target

if not warm_results:
    raise RuntimeError("warm_results is empty — run the warm-restart loop first.")

# Build the unified table: rows = traj, cols = targets + every divergence metric.
def _flux_rmse_proxy(res):
    """Relative RMSE between warm flux tail and GT-saturated tail. Independent
    proxy for "is the warm trajectory the right operating point?". Uses
    ref_flux_samples if available, else log_gt-tail."""
    lw = np.asarray(res["log_warm"]["eflux"])
    if lw.size == 0:
        return np.nan
    warm_tail = lw[-min(80, len(lw)):]
    ref = res.get("ref_flux_samples")
    if ref is None or len(ref) == 0:
        return np.nan
    ref = np.asarray(ref)
    return float(np.sqrt(np.mean((warm_tail.mean() - ref.mean()) ** 2 +
                                  (warm_tail.std() - ref.std()) ** 2))
                 / max(abs(ref.mean()), 1e-12))

rows = []
for traj, res in warm_results.items():
    if not res.get("div_warm"):
        continue
    row = {"traj": traj, "split": res.get("split", "ID")}
    if "per_iter_fids" in dir() and traj in per_iter_fids:
        for _lv, _v in per_iter_fids[traj].items():
            row[f"FID_{_lv}"] = _v
    row["flux_rmse"] = _flux_rmse_proxy(res)
    for k, v in res["div_warm"].items():
        if k.endswith("_error"): continue
        row[k] = v
    rows.append(row)

if len(rows) < 3:
    raise RuntimeError(f"only {len(rows)} trajs available — need ≥ 3 for Pearson r")

big_df = pd.DataFrame(rows)
print(f"Working with {len(big_df)} trajectories: "
      f"{(big_df['split']=='ID').sum()} ID + {(big_df['split']=='OOD').sum()} OOD\n")

target_cols = [c for c in big_df.columns
               if (c.startswith("FID_") or c == "flux_rmse")
                  and big_df[c].notna().sum() >= 3]
metric_cols = [c for c in big_df.columns
               if c not in (["traj", "split", "flux_rmse"] + target_cols)
                  and not c.startswith("FID_")
                  and big_df[c].notna().sum() >= 3
                  and big_df[c].std() > 1e-12 * (abs(big_df[c].mean()) + 1e-30)]
print(f"Targets:  {target_cols}")
print(f"Metrics:  {len(metric_cols)} (after pruning NaN / near-constant)\n")

def _pearson(x, y):
    x = np.asarray(x, dtype=float); y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 3 or np.std(x[mask]) == 0 or np.std(y[mask]) == 0:
        return np.nan
    return float(np.corrcoef(x[mask], y[mask])[0, 1])

# (1) Leaderboards per target -------------------------------------------------
for tgt in target_cols:
    rs = [(m, _pearson(big_df[m], big_df[tgt])) for m in metric_cols]
    rs = [(m, r) for m, r in rs if np.isfinite(r)]
    rs.sort(key=lambda x: -abs(x[1]))
    print(f"=== top {TOP_N} metrics most correlated with {tgt} ===")
    for m, r in rs[:TOP_N]:
        print(f"  {r:+.3f}   {m}")
    print()

# (2) Cross-family redundancy clusters ----------------------------------------
# Pearson r matrix across all kept metrics (across families).
K = len(metric_cols)
R = np.full((K, K), np.nan)
for i in range(K):
    for j in range(K):
        R[i, j] = _pearson(big_df[metric_cols[i]], big_df[metric_cols[j]])

# Greedy clustering: each metric joins the first cluster whose centroid has
# |r| ≥ R_REDUNDANT with it; otherwise starts a new cluster.
clusters = []
for i, m in enumerate(metric_cols):
    placed = False
    for cl in clusters:
        # representative: first member
        rep_idx = cl[0]
        if np.isfinite(R[i, rep_idx]) and abs(R[i, rep_idx]) >= R_REDUNDANT:
            cl.append(i); placed = True; break
    if not placed:
        clusters.append([i])

# Sort clusters by size (largest first), then by max |r| with the FIDs.
def _cluster_key(cl):
    rep = metric_cols[cl[0]]
    rs = [abs(_pearson(big_df[rep], big_df[t])) for t in target_cols]
    rs = [r for r in rs if np.isfinite(r)]
    return (-len(cl), -(max(rs) if rs else 0.0))
clusters.sort(key=_cluster_key)

print(f"=== {len(clusters)} redundancy clusters at |r| ≥ {R_REDUNDANT} ===\n")
for ci, cl in enumerate(clusters):
    rep = metric_cols[cl[0]]
    rep_rs = {t: _pearson(big_df[rep], big_df[t]) for t in target_cols}
    rep_rs_str = "  ".join(
        f"r({t})={rep_rs[t]:+.2f}" for t in target_cols if np.isfinite(rep_rs[t])
    )
    print(f"cluster {ci:2d}  (n={len(cl)})  rep = {rep}")
    print(f"             {rep_rs_str}")
    if len(cl) > 1:
        print(f"             redundant: {[metric_cols[k] for k in cl[1:]]}")
    print()

print("=== recommended canonical set (one per cluster, sorted by best target |r|) ===")
canonical = []
for cl in clusters:
    rep = metric_cols[cl[0]]
    best_r = max(
        (abs(_pearson(big_df[rep], big_df[t])) for t in target_cols),
        default=0.0,
    )
    canonical.append((rep, best_r, len(cl)))
canonical.sort(key=lambda x: -x[1])
for rep, best_r, n_cluster in canonical:
    print(f"  best |r|={best_r:.3f}  (cluster size {n_cluster:>2})  {rep}")

print(f"\nIf {R_REDUNDANT} is too aggressive (everything ends up in one cluster) drop it to "
      f"0.85; if too lenient (every metric is its own cluster) raise it to 0.97.")


In [ ]:
per_iter_fids = globals().get("per_iter_fids", {})
if not per_iter_fids:
    raise RuntimeError("run latent_dashboard cell first to populate per_iter_fids")
warm_results = globals().get("warm_results", {})
if not warm_results:
    raise RuntimeError("run cell 28 first to populate warm_results")

BASE_MEASURES = ["w1", "mmd", "ks_p", "ad", "arima_l2", "r2"]
FAMILIES      = ["flux", "ky_spec", "fluxspec"]

# What name does a (family, base) combo carry in warm_results[traj]["div_warm"]?
def _metric_name(family, base):
    if family == "flux":
        return f"flux_{base}_hist" if base == "r2" else f"flux_{base}"
    if family in ("ky_spec", "fluxspec"):
        return f"{family}_r2_meanlog" if base == "r2" else f"{family}_{base}_mean"
    return None

SEMANTICS = {
    "w1":       ("Wasserstein-1",         "physical units, transport cost",     "lower"),
    "mmd":      ("MMD-RBF²",              "smooth functional gap",                "lower"),
    "ks_p":     ("KS p-value",            "indistinguishability probability",    "higher"),
    "ad":       ("Anderson–Darling",      "tail-weighted shape disagreement",    "lower"),
    "arima_l2": ("ARIMA(2,0,2) param L2",  "dynamics / autocorrelation mismatch", "lower"),
    "r2":       ("R² (hist or meanlog)",  "point-wise reconstruction fidelity",  "higher"),
}

# Build a combined long DataFrame: one row per (traj, fid_level), wide cols per metric
rows = []
for traj, fids in per_iter_fids.items():
    if traj not in warm_results or not warm_results[traj].get("div_warm"):
        continue
    div = warm_results[traj]["div_warm"]
    for fid_lv, fid_v in fids.items():
        row = {"traj": traj, "fid_level": fid_lv, "FID": fid_v,
               "split": warm_results[traj].get("split", "ID")}
        for fam in FAMILIES:
            for base in BASE_MEASURES:
                key = _metric_name(fam, base)
                row[f"{fam}__{base}"] = div.get(key, np.nan)
        rows.append(row)

if not rows:
    raise RuntimeError("no overlap between per_iter_fids and warm_results[*]['div_warm']")

big = pd.DataFrame(rows)
fid_levels = sorted(big["fid_level"].unique())
print(f"Aggregating across {len(big['traj'].unique())} trajs × {len(fid_levels)} FID levels = "
      f"{len(big)} (traj, level) pairs.")

# Per (family, base, fid_level): Pearson r between the metric and FID across trajs.
def _pearson_safe(df, x_col, y_col):
    x = df[x_col].to_numpy(dtype=float); y = df[y_col].to_numpy(dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 3 or np.std(x[mask]) == 0 or np.std(y[mask]) == 0:
        return np.nan
    return float(np.corrcoef(x[mask], y[mask])[0, 1])

corr_cube = {}  # corr_cube[base][family][fid_level] = r
for base in BASE_MEASURES:
    corr_cube[base] = {}
    for fam in FAMILIES:
        corr_cube[base][fam] = {}
        col = f"{fam}__{base}"
        for fid_lv in fid_levels:
            sub = big[big["fid_level"] == fid_lv]
            corr_cube[base][fam][fid_lv] = _pearson_safe(sub, col, "FID")

# Score each base.
print(f"\n{'base':>10s}  {'mean|r|':>8s}  {'min|r|':>8s}  {'coverage':>8s}  "
      f"{'sign-consistent':>15s}  semantics")
print(f"{'-'*10}  {'-'*8}  {'-'*8}  {'-'*8}  {'-'*15}  {'-'*40}")
ranked = []
for base in BASE_MEASURES:
    rs = []
    fam_max_abs = {}
    for fam in FAMILIES:
        per_fid = [corr_cube[base][fam][lv] for lv in fid_levels]
        finite = [r for r in per_fid if np.isfinite(r)]
        if finite:
            rs.extend(finite)
            fam_max_abs[fam] = max(abs(r) for r in finite)
    if not rs:
        continue
    abs_rs = [abs(r) for r in rs]
    mean_abs = float(np.mean(abs_rs))
    min_abs_per_fam = min(fam_max_abs.values()) if fam_max_abs else 0.0
    coverage = sum(1 for v in fam_max_abs.values() if v >= 0.5)
    # sign consistency: do the family-best correlations all have the same sign?
    fam_best_signed = {}
    for fam in FAMILIES:
        per_fid = [corr_cube[base][fam][lv] for lv in fid_levels]
        finite = [(abs(r), r) for r in per_fid if np.isfinite(r)]
        if finite:
            finite.sort(reverse=True)
            fam_best_signed[fam] = finite[0][1]
    signs = {1 if v > 0 else -1 for v in fam_best_signed.values() if abs(v) >= 0.3}
    sign_consistent = "yes" if len(signs) <= 1 else "no"

    ranked.append({
        "base": base, "mean_abs": mean_abs, "min_per_fam_abs": min_abs_per_fam,
        "coverage": coverage, "sign_consistent": sign_consistent,
        "fam_best_signed": fam_best_signed,
    })

ranked.sort(key=lambda d: (-d["coverage"], -d["mean_abs"], -d["min_per_fam_abs"]))
for d in ranked:
    label, descr, dirn = SEMANTICS[d["base"]]
    print(f"{d['base']:>10s}  {d['mean_abs']:>8.3f}  {d['min_per_fam_abs']:>8.3f}  "
          f"{d['coverage']:>5d}/3  {d['sign_consistent']:>15s}  "
          f"{label:14s}  ({descr})")

# Per-base, per-family, per-fid-level table for the top recommendation.
print("\n=== top recommendation: detailed (family × FID-level) Pearson r ===")
top = ranked[0]["base"]
label, descr, dirn = SEMANTICS[top]
print(f"\n  best base measure : {top}  ({label}, {descr}; {dirn} = better)\n")
header = f"  {'family':>10s}  " + "  ".join(f"{lv:>14s}" for lv in fid_levels)
print(header)
for fam in FAMILIES:
    row = corr_cube[top][fam]
    cells = "  ".join(
        (f"{row[lv]:>+14.3f}" if np.isfinite(row[lv]) else f"{'nan':>14s}")
        for lv in fid_levels
    )
    print(f"  {fam:>10s}  {cells}")

print(f"\nIn the paper, report {top} consistently as flux_{top}{'_hist' if top=='r2' else ''} / "
      f"ky_spec_{top}{'_meanlog' if top=='r2' else '_mean'} / fluxspec_{top}"
      f"{'_meanlog' if top=='r2' else '_mean'} alongside the FID columns. ")
print(f"Direction: {dirn} = better warm-start. ")
if ranked[0]["sign_consistent"] != "yes":
    print("WARNING: sign of correlation is not consistent across families — flag this in caption.")

In [ ]:
import numpy as np

apply_paper_style()
TOP_BASE = ranked[0]["base"]
LEVEL_COLOR = {
    "skip_deep":  darken(COLOR_QY, 0.5),
    "bottleneck": COLOR_QY,
    "flux_head":  COLOR_FLUX,
    "phi":        COLOR_KY,
}
fams = ["flux", "ky_spec", "fluxspec"]


def _norm01(a):
    a = np.asarray(a, dtype=float)
    f = a[np.isfinite(a)]
    if f.size == 0 or f.max() == f.min():
        return np.zeros_like(a)
    return (a - f.min()) / (f.max() - f.min())


fig, axes = plt.subplots(1, len(fams), figsize=(3.2 * len(fams), 3.4),
                          sharex=True, sharey=True)
PAPER_METRIC_DATA = {"top_base": TOP_BASE, "fid_levels": list(fid_levels),
                     "panels": {}}

for col, fam in enumerate(fams):
    ax = axes[col]
    panel = {"per_level": {}}
    metric_col = f"{fam}__{TOP_BASE}"
    metric_vals = big[metric_col].to_numpy(dtype=float)
    for lv in fid_levels:
        sub = big[big["fid_level"] == lv]
        m = sub[metric_col].to_numpy(dtype=float)
        f = sub["FID"].to_numpy(dtype=float)
        ok = np.isfinite(m) & np.isfinite(f)
        if ok.sum() < 2:
            continue
        x = _norm01(f[ok])
        y = _norm01(m[ok])
        r = corr_cube[TOP_BASE][fam][lv]
        c = LEVEL_COLOR.get(lv, "#666")
        ax.scatter(x, y, s=22, color=c, edgecolor="none", alpha=0.85,
                    label=f"{lv}  r={r:+.2f}" if np.isfinite(r) else lv)
        panel["per_level"][lv] = {"fid_norm": x, "metric_norm": y,
                                   "fid_raw": f[ok], "metric_raw": m[ok], "r": r}
    ax.plot([0, 1], [0, 1], color=COLOR_REF, lw=0.8, ls="--", alpha=0.4)
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-0.05, 1.05)
    ax.set_title(fam, fontsize=9)
    if col == 0:
        ax.set_ylabel(f"{TOP_BASE} (norm.)")
    ax.set_xlabel("FID (norm.)")
    ax.legend(fontsize=7, loc="upper left", frameon=False)
    PAPER_METRIC_DATA["panels"][fam] = panel

fig.suptitle(f"Best paper metric: {TOP_BASE}  vs  FID  (4 U-Net levels)",
              fontsize=10, y=1.02)
fig.tight_layout()
paths = save_fig(fig, f"metric_vs_fid_{TOP_BASE}", RESULTS_DIR)
print(f"saved: {paths}")
plt.show()

In [ ]:
import pandas as pd
import numpy as np

# 4-level FID per trajectory
fid_rows = []
for traj, fids in per_iter_fids.items():
    row = {"traj": traj, "split": warm_results.get(traj, {}).get("split", "ID")}
    row.update(fids)
    fid_rows.append(row)
fid_df = pd.DataFrame(fid_rows).set_index("traj").sort_index()
print("=== FIDs at 4 U-Net levels per trajectory ===")
print(fid_df.to_string(float_format=lambda x: f"{x:.4f}"))

# Per-traj distributional metrics (warm vs GT) for the canonical 6 base measures
METRIC_KEYS = {
    "flux":     {"mmd": "flux_mmd",          "ad": "flux_ad",          "ks_p": "flux_ks_p",
                  "w1": "flux_w1",            "arima_l2": "flux_arima_l2",
                  "r2": "flux_r2_hist"},
    "ky_spec":  {"mmd": "ky_spec_mmd_mean",  "ad": "ky_spec_ad_mean",  "ks_p": "ky_spec_ks_p_mean",
                  "w1": "ky_spec_w1_mean",    "arima_l2": "ky_spec_arima_l2_mean",
                  "r2": "ky_spec_r2_meanlog"},
    "fluxspec": {"mmd": "fluxspec_mmd_mean", "ad": "fluxspec_ad_mean", "ks_p": "fluxspec_ks_p_mean",
                  "w1": "fluxspec_w1_mean",   "arima_l2": "fluxspec_arima_l2_mean",
                  "r2": "fluxspec_r2_meanlog"},
}

div_rows = []
for traj, res in warm_results.items():
    div = res.get("div_warm", {}) or {}
    row = {"traj": traj, "split": res.get("split", "ID")}
    for fam, kmap in METRIC_KEYS.items():
        for short, key in kmap.items():
            row[f"{fam}_{short}"] = div.get(key, np.nan)
    div_rows.append(row)
div_df = pd.DataFrame(div_rows).set_index("traj").sort_index()
print("\n=== distributional metrics (warm vs GT) ===")
print(div_df.to_string(float_format=lambda x: f"{x:.4g}"))

# Aggregated summary: mean over trajectories per (split, family, measure)
SUMMARY_MEASURES = ["mmd", "ad", "ks_p", "w1", "arima_l2", "r2"]
summary_rows = []
for split in ("ID", "OOD", "ALL"):
    sub = div_df if split == "ALL" else div_df[div_df["split"] == split]
    if sub.empty:
        continue
    row = {"split": split, "n": len(sub)}
    for fam in METRIC_KEYS:
        for m in SUMMARY_MEASURES:
            col = f"{fam}_{m}"
            if col in sub:
                row[f"{fam}_{m}"] = float(sub[col].mean(skipna=True))
    summary_rows.append(row)
summary_df = pd.DataFrame(summary_rows).set_index("split")
print("\n=== summary (mean per split) ===")
print(summary_df.to_string(float_format=lambda x: f"{x:.4g}"))

# Also: FID summary per split.
fid_summary = (fid_df.reset_index()
                  .groupby("split")[list(per_iter_fids[next(iter(per_iter_fids))].keys())]
                  .mean())
print("\n=== FID summary (mean per split) ===")
print(fid_summary.to_string(float_format=lambda x: f"{x:.4f}"))

# Persist the raw per-traj tables (CSV for inspection, plus they go into the
# pickle dump in the next cell).
import os
os.makedirs(RESULTS_DIR, exist_ok=True)
fid_df.to_csv(os.path.join(RESULTS_DIR, f"{METHOD_NAME}_fid.csv"))
div_df.to_csv(os.path.join(RESULTS_DIR, f"{METHOD_NAME}_div.csv"))
summary_df.to_csv(os.path.join(RESULTS_DIR, f"{METHOD_NAME}_summary.csv"))
fid_summary.to_csv(os.path.join(RESULTS_DIR, f"{METHOD_NAME}_fid_summary.csv"))
print(f"\nwrote per-method CSVs under {RESULTS_DIR}")


## Paper figure — qualitative warm-restart overview

Single composite figure per ID trajectory:

* **(a)** $W(k_y)$ heatmap over time for the warm trajectory (warm-start initial
  condition is at $t=0$, the trajectory is run forward by `N_EVAL_STEPS`).
* **(b)** Flux $Q(t)$: full GT trajectory from $t=0$ (transient + saturated)
  vs the *warm* trajectory which starts already in saturation.
* **(c)** Time-saved bar: GKW transient steps (proxied by the steps before
  the GT running mean reaches the saturation band) vs warm-start (zero
  transient by construction).


In [ ]:
import numpy as np
import os
from matplotlib.gridspec import GridSpec

apply_paper_style()
QUAL_TRAJS = [t for t in warm_results if not t.startswith("ood_")][:3]
GKW_DT = 0.4   # raw GKW step in R/v_th
PAPER_QUAL = {}

for traj in QUAL_TRAJS:
    res = warm_results[traj]
    log_warm = res.get("log_warm", {})
    log_gt   = res.get("log_gt_ref", {})
    if "eflux" not in log_warm:
        print(f"  skip {traj}: no warm log")
        continue

    ky_warm = np.asarray(log_warm.get("ky_spec", []))
    ky_gt   = np.asarray(log_gt.get("ky_spec", []))
    warm_flux = np.asarray(log_warm["eflux"], dtype=float)
    t_warm = np.asarray(log_warm.get("time", []), dtype=float)

    # GT flux + time from raw files (works for ID + OOD).
    gkw_path = (os.path.join(GKW_RAW_DIR, "ood", traj[len("ood_"):])
                if traj.startswith("ood_iteration_")
                else os.path.join(GKW_RAW_DIR, traj))
    gt_full = None; t_gt = None
    p = os.path.join(gkw_path, "fluxes.dat")
    if os.path.isfile(p):
        gt_full = np.asarray(np.loadtxt(p)[:, 1], dtype=float)
    p = os.path.join(gkw_path, "time.dat")
    if os.path.isfile(p):
        t_gt = np.asarray(np.loadtxt(p), dtype=float)
    elif gt_full is not None:
        t_gt = np.arange(len(gt_full)) * GKW_DT

    fig = plt.figure(figsize=(7.6, 3.6))
    gs = GridSpec(2, 2, figure=fig, height_ratios=[1.0, 1.0],
                  width_ratios=[1.6, 1.0], hspace=0.55, wspace=0.35)
    ax_ky    = fig.add_subplot(gs[0, 0])
    ax_flux  = fig.add_subplot(gs[1, 0])
    ax_spec  = fig.add_subplot(gs[:, 1])

    # (a) warm ky heatmap — x in physical time
    if ky_warm.ndim == 2 and ky_warm.size > 0:
        T = ky_warm.shape[0]
        x_extent = (float(t_warm[0]) if t_warm.size == T else 0.0,
                    float(t_warm[-1]) if t_warm.size == T else float(T))
        im = ax_ky.imshow(np.log10(np.maximum(ky_warm.T, 1e-30)),
                            aspect="auto", origin="lower", cmap="magma",
                            extent=[x_extent[0], x_extent[1], 0, ky_warm.shape[1]])
        ax_ky.set_xlabel(r"time $[v_{th}/R]$")
        ax_ky.set_ylabel(r"$k_y$")
        fig.colorbar(im, ax=ax_ky, fraction=0.025, pad=0.01)

    # (b) flux: warm + GT both on the SAME physical-time axis
    if gt_full is not None and t_gt is not None:
        ax_flux.plot(t_gt, gt_full, color=COLOR_REF, lw=0.9, alpha=0.7, label="GKW")
    if t_warm.size == len(warm_flux):
        ax_flux.plot(t_warm, warm_flux, color=COLOR_GYROFLOW, lw=1.3, label="warm")
    else:
        ax_flux.plot(np.arange(len(warm_flux)) * GKW_DT, warm_flux,
                       color=COLOR_GYROFLOW, lw=1.3, label="warm (assumed dt=0.4)")
    ax_flux.set_xlabel(r"time $[v_{th}/R]$")
    ax_flux.set_ylabel(r"$Q$")
    ax_flux.legend(fontsize=7, loc="best", frameon=False)

    # (c) time-averaged log-W(k_y) — warm vs GT-saturated tail
    if ky_warm.ndim == 2 and ky_warm.size and ky_gt.ndim == 2 and ky_gt.size:
        warm_mean = ky_warm.mean(axis=0)
        gt_mean   = ky_gt.mean(axis=0)
        kk = np.arange(len(warm_mean))
        ax_spec.semilogy(kk, np.maximum(gt_mean, 1e-30), color=COLOR_REF,
                          lw=1.0, label="GKW (sat.)")
        ax_spec.semilogy(kk, np.maximum(warm_mean, 1e-30), color=COLOR_GYROFLOW,
                          lw=1.4, label="warm")
        ax_spec.set_xlabel(r"$k_y$")
        ax_spec.set_ylabel(r"$\langle W(k_y) \rangle$")
        ax_spec.legend(fontsize=7, loc="best", frameon=False)
    else:
        ax_spec.text(0.5, 0.5, "no spectrum", transform=ax_spec.transAxes,
                       ha="center", va="center", fontsize=8)

    fig.suptitle(traj, fontsize=9, y=1.02)
    fig.tight_layout()
    paths = save_fig(fig, f"warm_overview_{traj}", RESULTS_DIR)
    print(f"  saved {traj}: {paths}")
    plt.show()

    PAPER_QUAL[traj] = {
        "warm_eflux":   warm_flux,
        "warm_time":    t_warm,
        "gt_full_flux": gt_full,
        "gt_time":      t_gt,
        "warm_kyspec":  ky_warm,
        "gt_kyspec":    ky_gt,
    }


In [ ]:
per_iter_fids = globals().get("per_iter_fids", {})
warm_results  = globals().get("warm_results", {})
if not per_iter_fids or not warm_results:
    raise RuntimeError("run cell 28 + latent_dashboard first")

LEVELS = {
    "bottleneck": ("middle block",      "#2a9d8f"),
    "phi":        (r"$\phi$ decoder",   "#8a3ffc"),
}
FAMILY_LABELS = {"ky_spec": r"$W(k_y)$", "fluxspec": r"$Q(k_y)$"}

MEASURES_TO_PLOT = {
    "Anderson–Darling": {"ky_spec": "ky_spec_ad_mean",  "fluxspec": "fluxspec_ad_mean"},
    "MMD":              {"ky_spec": "ky_spec_mmd_mean", "fluxspec": "fluxspec_mmd_mean"},
}

def _norm01(arr):
    a = np.asarray(arr, dtype=float)
    finite = a[np.isfinite(a)]
    if finite.size == 0 or finite.max() == finite.min():
        return np.zeros_like(a)
    lo, hi = finite.min(), finite.max()
    return (a - lo) / (hi - lo)

trajs = [t for t in per_iter_fids if t in warm_results
          and warm_results[t].get("div_warm")]

apply_paper_style()
all_rows = []
for meas_label, fam_keys in MEASURES_TO_PLOT.items():
    fig, axes = plt.subplots(1, len(fam_keys),
                              figsize=(3.6 * len(fam_keys), 3.4),
                              sharey=True, squeeze=False)
    for col_i, (fam, mkey) in enumerate(fam_keys.items()):
        ax = axes[0, col_i]
        meas_vals = np.array([warm_results[t]["div_warm"].get(mkey, np.nan)
                               for t in trajs], dtype=float)
        meas_n = _norm01(meas_vals)
        for lv, (lv_label, lv_color) in LEVELS.items():
            fid_v = np.array([per_iter_fids[t].get(lv, np.nan) for t in trajs], dtype=float)
            fid_n = _norm01(fid_v)
            for j, t in enumerate(trajs):
                ax.scatter(fid_n[j], meas_n[j], s=24, facecolor=lv_color, edgecolor=lv_color, lw=0.8, marker="o", alpha=0.9)
            mask = np.isfinite(fid_v) & np.isfinite(meas_vals)
            if mask.sum() >= 3 and np.std(fid_v[mask]) > 0 and np.std(meas_vals[mask]) > 0:
                r = float(np.corrcoef(fid_v[mask], meas_vals[mask])[0, 1])
                r2 = r ** 2
            else:
                r2 = np.nan
            all_rows.append({"measure": meas_label, "family": fam, "level": lv, "r2": r2})
            label = f"{lv_label}  R²={r2:.2f}" if np.isfinite(r2) else lv_label
            ax.scatter([], [], s=24, color=lv_color, edgecolor=lv_color, label=label)
        ax.plot([0, 1], [0, 1], "k--", lw=0.8, alpha=0.35)
        ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.05, 1.05)
        ax.set_xlabel("normalized FID")
        if col_i == 0:
            ax.set_ylabel(f"normalized {meas_label}")
        ax.set_title(FAMILY_LABELS[fam], fontsize=10)
        ax.grid(alpha=0.25)
        ax.legend(fontsize=8, loc="upper left", frameon=False)
    fig.suptitle(rf"{meas_label}  vs  FID — middle block / $\phi$ decoder",
                  fontsize=10, y=1.02)
    fig.tight_layout()
    slug = "ad" if meas_label.startswith("Ander") else "mmd"
    paths = save_fig(fig, f"{slug}_vs_fid_middle_phi", RESULTS_DIR)
    print(f"saved {meas_label}: {paths}")
    plt.show()

## Persist results for the cross-method comparison notebook

Dumps a single pickle keyed by `METHOD_NAME` containing FIDs, distributional
metrics, summary tables, and qualitative arrays. Loaded by
`neurips_compare_methods.ipynb` to render multi-method tables/plots in the
same paper style.

In [ ]:
# === persist all paper-figure data for the cross-method notebook ===========
import numpy as np

ID_trajs  = [t for t in warm_results if not t.startswith("ood_")]
OOD_trajs = [t for t in warm_results if     t.startswith("ood_")]

payload = {
    "method":       METHOD_NAME,
    "n_traj":       len(warm_results),
    "ID_trajs":     ID_trajs,
    "OOD_trajs":    OOD_trajs,
    "fids":         {lv: {t: float(per_iter_fids[t][lv])
                            for t in per_iter_fids if lv in per_iter_fids[t]}
                       for lv in {lv for d in per_iter_fids.values() for lv in d}},
    "metrics":      {col: div_df[col].to_dict() for col in div_df.columns
                       if col != "split"},
    "summary":      summary_df.to_dict(),
    "fid_summary":  fid_summary.to_dict(),
    "probe":        PAPER_PROBE if "PAPER_PROBE" in dir() else {},
    "metric_corr":  PAPER_METRIC_DATA if "PAPER_METRIC_DATA" in dir() else {},
    "qualitative": {
        "warm_eflux":   {t: PAPER_QUAL[t]["warm_eflux"]   for t in PAPER_QUAL},
        "gt_full_flux": {t: PAPER_QUAL[t]["gt_full_flux"] for t in PAPER_QUAL},
        "warm_kyspec":  {t: PAPER_QUAL[t]["warm_kyspec"]  for t in PAPER_QUAL},
        "gt_kyspec":    {t: PAPER_QUAL[t].get("gt_kyspec") for t in PAPER_QUAL},
    },
    "config": {
        "N_EVAL_STEPS":      N_EVAL_STEPS,
        "N_DENOISING_STEPS": N_DENOISING_STEPS,
        "FID_N_COMPONENTS":  FID_N_COMPONENTS,
        "PRETRAINED_DIR":    PRETRAINED_DIR,
        "AE_CHECKPOINT":     AE_CHECKPOINT,
        "GYROSWIN_CHECKPOINT": GYROSWIN_CHECKPOINT,
    },
}
path = save_method_results(METHOD_NAME, payload, RESULTS_DIR)
print(f"persisted method '{METHOD_NAME}' results to {path}")
print(f"keys: {sorted(payload.keys())}")
